# <center>Deep Generative Models</center>
## <center>Seminar 12 — Video Diffusion: From Image to Video Generation</center>

<center><b>April 23, 2026</b></center>

**Plan**:
1. From image diffusion to video diffusion: temporal attention, 3D UNet vs DiT (15 min)
2. Architecture overview: Sora, Kling, CogVideoX, HunyuanVideo, Mochi, LTX, Wan, Veo (20 min)
3. Autoregressive video: next-frame prediction, causal attention, world models (15 min)
4. Practice: short video generation with Wan 2.1-1.3B / SVD / LTX (30 min)
5. Open problems: temporal consistency, long videos, motion control (7 min)
6. Summary and connection to the course (3 min)

> This seminar directly builds on **Seminar 11** (Evolution of Stable Diffusion). The main thesis: video diffusion in 2024–2025 recapitulates image diffusion's 2022–2023 evolution with a ~6-month lag — the same four axes (architecture, VAE, text encoder, objective) played out again, this time on 4D tensors.

In [ ]:
!pip install -q -U diffusers transformers accelerate safetensors sentencepiece protobuf
!pip install -q imageio imageio-ffmpeg mediapy ftfy
# bitsandbytes — needed to 4-bit-quantize Wan's UMT5-XXL text encoder on T4:
!pip install -q bitsandbytes

import torch
import gc
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm
import mediapy as media
from diffusers.utils import export_to_video, load_image
from IPython.display import HTML
import warnings
warnings.filterwarnings('ignore')

assert torch.cuda.is_available(), "GPU required"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

SEED = 42

## 1. From Image to Video Diffusion — Theory Bridge

This section connects everything we learned about image diffusion (Seminars 9–11, Lectures 11–12) with the video setting. We will see that **the four axes of evolution are the same** — they are just played out on one-dimension-larger tensors.

<center>
<svg width="700" height="150" xmlns="http://www.w3.org/2000/svg" font-family="Arial, sans-serif" font-size="11">
  <text x="350" y="18" text-anchor="middle" font-weight="bold" font-size="13" fill="#333">Roadmap — four axes we will revisit on video</text>
  <g transform="translate(20,40)">
    <rect x="0"   y="0" width="150" height="90" rx="8" fill="#e8f0fe" stroke="#4285f4" stroke-width="1.5"/>
    <text x="75" y="22" text-anchor="middle" fill="#1a73e8" font-weight="bold" font-size="12">① Backbone</text>
    <text x="75" y="50" text-anchor="middle" fill="#555" font-size="10">UNet → DiT → MMDiT</text>
    <text x="75" y="68" text-anchor="middle" fill="#555" font-size="10">→ dual/single stream</text>
    <text x="75" y="85" text-anchor="middle" fill="#888" font-size="9">§1.2–1.4 · §2</text>
  </g>
  <g transform="translate(190,40)">
    <rect x="0"   y="0" width="150" height="90" rx="8" fill="#fef7e0" stroke="#f9ab00" stroke-width="1.5"/>
    <text x="75" y="22" text-anchor="middle" fill="#e37400" font-weight="bold" font-size="12">② VAE</text>
    <text x="75" y="50" text-anchor="middle" fill="#555" font-size="10">2D → 3D causal</text>
    <text x="75" y="68" text-anchor="middle" fill="#555" font-size="10">→ extreme 1:192 (LTX)</text>
    <text x="75" y="85" text-anchor="middle" fill="#888" font-size="9">§1.5 · §2.6</text>
  </g>
  <g transform="translate(360,40)">
    <rect x="0"   y="0" width="150" height="90" rx="8" fill="#fce4ec" stroke="#e91e63" stroke-width="1.5"/>
    <text x="75" y="22" text-anchor="middle" fill="#880e4f" font-weight="bold" font-size="12">③ Text encoder</text>
    <text x="75" y="50" text-anchor="middle" fill="#555" font-size="10">T5 → MLLM</text>
    <text x="75" y="68" text-anchor="middle" fill="#555" font-size="10">(llava-llama-3-8b)</text>
    <text x="75" y="85" text-anchor="middle" fill="#888" font-size="9">§2.4</text>
  </g>
  <g transform="translate(530,40)">
    <rect x="0"   y="0" width="150" height="90" rx="8" fill="#e8f5e9" stroke="#34a853" stroke-width="1.5"/>
    <text x="75" y="22" text-anchor="middle" fill="#1b5e20" font-weight="bold" font-size="12">④ Objective</text>
    <text x="75" y="50" text-anchor="middle" fill="#555" font-size="10">DDPM → Flow Matching</text>
    <text x="75" y="68" text-anchor="middle" fill="#555" font-size="10">universal in 2024+</text>
    <text x="75" y="85" text-anchor="middle" fill="#888" font-size="9">§1.4 · §2.9</text>
  </g>
</svg>
</center>

### 1.1 Why Video is Hard

A video is a **4D tensor**:
$$\mathbf{x} \in \mathbb{R}^{T \times H \times W \times C}$$

<center>
<svg width="680" height="230" xmlns="http://www.w3.org/2000/svg" font-family="Arial, sans-serif" font-size="12">
  <defs><marker id="a1" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#666"/></marker></defs>
  <text x="340" y="20" text-anchor="middle" font-size="14" font-weight="bold" fill="#333">A video is a 4D tensor — three constraints follow</text>
  <!-- Frame stack visualization -->
  <g transform="translate(30,50)">
    <rect x="0"  y="0"  width="110" height="80" rx="3" fill="#e8f0fe" stroke="#4285f4" stroke-width="1.5"/>
    <rect x="8"  y="8"  width="110" height="80" rx="3" fill="#e8f0fe" stroke="#4285f4" stroke-width="1.5"/>
    <rect x="16" y="16" width="110" height="80" rx="3" fill="#e8f0fe" stroke="#4285f4" stroke-width="1.5"/>
    <rect x="24" y="24" width="110" height="80" rx="3" fill="#e8f0fe" stroke="#4285f4" stroke-width="1.5"/>
    <rect x="32" y="32" width="110" height="80" rx="3" fill="#e8f0fe" stroke="#4285f4" stroke-width="1.5"/>
    <text x="87" y="75" text-anchor="middle" fill="#1a73e8" font-weight="bold" font-size="11">H × W × C</text>
    <text x="87" y="130" text-anchor="middle" fill="#555" font-size="11">T frames</text>
    <line x1="0" y1="115" x2="150" y2="115" stroke="#999" stroke-width="1" marker-end="url(#a1)"/>
    <text x="75" y="152" text-anchor="middle" fill="#888" font-size="10" font-style="italic">time</text>
  </g>
  <!-- three challenges -->
  <g transform="translate(220,55)">
    <rect x="0"   y="0"  width="140" height="45" rx="6" fill="#fce8e6" stroke="#ea4335" stroke-width="1.5"/>
    <text x="70"  y="18" text-anchor="middle" fill="#c5221f" font-weight="bold" font-size="11">① Frame-by-frame</text>
    <text x="70"  y="34" text-anchor="middle" fill="#555" font-size="10">= slideshow, flickers</text>
    <rect x="150" y="0"  width="140" height="45" rx="6" fill="#fef7e0" stroke="#f9ab00" stroke-width="1.5"/>
    <text x="220" y="18" text-anchor="middle" fill="#e37400" font-weight="bold" font-size="11">② Full 3D attention</text>
    <text x="220" y="34" text-anchor="middle" fill="#555" font-size="10">O((T·H·W)²) — ≈10¹¹</text>
    <rect x="300" y="0"  width="140" height="45" rx="6" fill="#e6f4ea" stroke="#34a853" stroke-width="1.5"/>
    <text x="370" y="18" text-anchor="middle" fill="#1b5e20" font-weight="bold" font-size="11">③ Data scarcity</text>
    <text x="370" y="34" text-anchor="middle" fill="#555" font-size="10">2–3 orders &lt; image-text</text>
    <text x="220" y="75" text-anchor="middle" fill="#444" font-size="11" font-style="italic">Every design below answers at least one of these.</text>
  </g>
</svg>
</center>

Three problems appear the moment we add the time axis:

1. **Naive per-frame generation is unusable.** Running a frozen SD 1.5 on every frame with the same seed gives a slideshow with flickering textures and morphing faces — nothing ties neighbouring frames together.

2. **Full 3D attention is infeasible.** An attention block over all spatio-temporal tokens costs $O((T \cdot H \cdot W)^2)$ — for a 5-second 720p clip that is $\sim 10^{11}$ attention entries per head, per layer.

3. **Data scarcity.** High-quality video-text pairs are 2–3 orders of magnitude less abundant than image-text pairs. Every design choice has to make video-native training cheap enough to reach scale.

Every recipe in this seminar is a response to at least one of these three constraints.

### 1.2 Temporal Attention — Factorized Space-Time (the VDM recipe)

The first successful design — **Video Diffusion Models** (Ho et al. 2022) — starts from an image U-Net and adds time by **factorizing** the attention.

**Step-by-step construction** (Ho et al. 2022):
- Take a 2D U-Net for images.
- Replace 2D convolutions (`3×3`) with **space-only 3D convolutions** (`1×3×3`). Frames are treated as a batch dimension — no information flows across time in convolutions.
- Insert a **temporal attention block** after each spatial attention block.

The two blocks differ only in how the batch and sequence axes are arranged:

<center>
<svg width="660" height="310" xmlns="http://www.w3.org/2000/svg" font-family="Arial, sans-serif" font-size="12">
  <defs><marker id="a2" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#666"/></marker></defs>
  <!-- Spatial block -->
  <g transform="translate(10,10)">
    <rect x="0" y="0" width="300" height="280" rx="10" fill="#f7faff" stroke="#4285f4" stroke-width="1.5"/>
    <text x="150" y="22" text-anchor="middle" font-size="13" font-weight="bold" fill="#1a73e8">Spatial block</text>
    <text x="150" y="40" text-anchor="middle" fill="#555" font-size="11">frames in batch, attention over pixels</text>
    <!-- mini frame grid -->
    <g transform="translate(20,60)">
      <rect x="0"  y="0"  width="55" height="55" fill="#cfe2ff" stroke="#4285f4"/>
      <rect x="60" y="0"  width="55" height="55" fill="#cfe2ff" stroke="#4285f4"/>
      <rect x="120"  y="0"  width="55" height="55" fill="#cfe2ff" stroke="#4285f4"/>
      <rect x="180" y="0"  width="55" height="55" fill="#cfe2ff" stroke="#4285f4"/>
      <text x="27" y="34" text-anchor="middle" fill="#1a73e8" font-size="10">frame 1</text>
      <text x="87" y="34" text-anchor="middle" fill="#1a73e8" font-size="10">frame 2</text>
      <text x="147" y="34" text-anchor="middle" fill="#1a73e8" font-size="10">frame 3</text>
      <text x="207" y="34" text-anchor="middle" fill="#1a73e8" font-size="10">frame 4</text>
      <text x="117" y="75" text-anchor="middle" fill="#1a73e8" font-size="10" font-style="italic">independent attention per frame</text>
      <!-- attention arrows within one frame -->
      <g transform="translate(0,95)">
        <rect x="60" y="0" width="55" height="55" fill="#e8f0fe" stroke="#4285f4"/>
        <circle cx="75"  cy="12" r="3" fill="#1a73e8"/>
        <circle cx="95"  cy="12" r="3" fill="#1a73e8"/>
        <circle cx="75"  cy="28" r="3" fill="#1a73e8"/>
        <circle cx="95"  cy="28" r="3" fill="#1a73e8"/>
        <circle cx="75"  cy="44" r="3" fill="#1a73e8"/>
        <circle cx="95"  cy="44" r="3" fill="#1a73e8"/>
        <line x1="75"  y1="12" x2="95" y2="44" stroke="#1a73e8" stroke-width="0.7" opacity="0.6"/>
        <line x1="95"  y1="12" x2="75" y2="44" stroke="#1a73e8" stroke-width="0.7" opacity="0.6"/>
        <line x1="75"  y1="12" x2="95" y2="28" stroke="#1a73e8" stroke-width="0.7" opacity="0.6"/>
        <line x1="95"  y1="12" x2="75" y2="28" stroke="#1a73e8" stroke-width="0.7" opacity="0.6"/>
        <text x="150" y="32" fill="#333" font-size="11">sequence = H·W pixels</text>
        <text x="150" y="50" fill="#888" font-size="10" font-style="italic">cost = O(N²) per frame</text>
      </g>
    </g>
    <rect x="40" y="235" width="220" height="30" rx="4" fill="#e8f0fe" stroke="#4285f4"/>
    <text x="150" y="254" text-anchor="middle" font-weight="bold" fill="#1a73e8" font-size="11">total: O(N² · T)</text>
  </g>
  <!-- Temporal block -->
  <g transform="translate(350,10)">
    <rect x="0" y="0" width="300" height="280" rx="10" fill="#fff9f0" stroke="#f9ab00" stroke-width="1.5"/>
    <text x="150" y="22" text-anchor="middle" font-size="13" font-weight="bold" fill="#e37400">Temporal block</text>
    <text x="150" y="40" text-anchor="middle" fill="#555" font-size="11">pixels in batch, attention over time</text>
    <!-- pixel-over-time visualisation -->
    <g transform="translate(20,65)">
      <circle cx="10"  cy="10" r="5" fill="#ffb74d" stroke="#e65100"/>
      <circle cx="70"  cy="10" r="5" fill="#ffb74d" stroke="#e65100"/>
      <circle cx="130" cy="10" r="5" fill="#ffb74d" stroke="#e65100"/>
      <circle cx="190" cy="10" r="5" fill="#ffb74d" stroke="#e65100"/>
      <circle cx="250" cy="10" r="5" fill="#ffb74d" stroke="#e65100"/>
      <line x1="10"  y1="10" x2="70" y2="10" stroke="#e65100" stroke-width="0.8" opacity="0.8"/>
      <line x1="70"  y1="10" x2="130" y2="10" stroke="#e65100" stroke-width="0.8" opacity="0.8"/>
      <line x1="130" y1="10" x2="190" y2="10" stroke="#e65100" stroke-width="0.8" opacity="0.8"/>
      <line x1="190" y1="10" x2="250" y2="10" stroke="#e65100" stroke-width="0.8" opacity="0.8"/>
      <text x="10"  y="30" text-anchor="middle" font-size="9" fill="#888">t₁</text>
      <text x="70"  y="30" text-anchor="middle" font-size="9" fill="#888">t₂</text>
      <text x="130" y="30" text-anchor="middle" font-size="9" fill="#888">t₃</text>
      <text x="190" y="30" text-anchor="middle" font-size="9" fill="#888">t₄</text>
      <text x="250" y="30" text-anchor="middle" font-size="9" fill="#888">t₅</text>
      <text x="130" y="55" text-anchor="middle" fill="#e37400" font-size="10" font-style="italic">one pixel trajectory over time</text>
    </g>
    <!-- attention arrows over time -->
    <g transform="translate(40,140)">
      <circle cx="10" cy="10" r="4" fill="#e65100"/>
      <circle cx="60" cy="10" r="4" fill="#e65100"/>
      <circle cx="110" cy="10" r="4" fill="#e65100"/>
      <circle cx="160" cy="10" r="4" fill="#e65100"/>
      <circle cx="210" cy="10" r="4" fill="#e65100"/>
      <path d="M 10 10 Q 85 -18 160 10" fill="none" stroke="#e65100" stroke-width="0.9" opacity="0.7"/>
      <path d="M 10 10 Q 60 -8 110 10" fill="none" stroke="#e65100" stroke-width="0.9" opacity="0.7"/>
      <path d="M 60 10 Q 135 -18 210 10" fill="none" stroke="#e65100" stroke-width="0.9" opacity="0.7"/>
      <text x="110" y="38" text-anchor="middle" fill="#333" font-size="11">sequence = T frames</text>
      <text x="110" y="55" text-anchor="middle" fill="#888" font-size="10" font-style="italic">cost = O(T²) per pixel</text>
    </g>
    <rect x="40" y="235" width="220" height="30" rx="4" fill="#fef7e0" stroke="#f9ab00"/>
    <text x="150" y="254" text-anchor="middle" font-weight="bold" fill="#e37400" font-size="11">total: O(N · T²)</text>
  </g>
</svg>
</center>

Total attention cost with $N = H \cdot W$:

$$\underbrace{O(N^2 T)}_{\text{spatial}} + \underbrace{O(N T^2)}_{\text{temporal}} \ll \underbrace{O(N^2 T^2)}_{\text{full 3D}}$$

[**Accent**] If temporal attention is disabled, the model collapses to an independent image model over frames. This lets us **train jointly on images + videos** — since high-quality video-text pairs are scarce but image-text pairs are abundant, this trick was the unlock for every 2022–2023 video model.

### 1.3 Family Tree — Temporal Layers in Image Models

The same factorized recipe powers the pre-transformer generation:

<center>
<svg width="700" height="230" xmlns="http://www.w3.org/2000/svg" font-family="Arial, sans-serif" font-size="11">
  <defs><marker id="a3" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#999"/></marker></defs>
  <text x="350" y="18" text-anchor="middle" font-weight="bold" font-size="13" fill="#333">Timeline of U-Net-based video models (2022–2024)</text>
  <!-- Axis -->
  <line x1="40" y1="180" x2="660" y2="180" stroke="#bbb" stroke-width="1"/>
  <text x="40"  y="205" text-anchor="middle" fill="#888" font-size="11">2022</text>
  <text x="265" y="205" text-anchor="middle" fill="#888" font-size="11">2023</text>
  <text x="490" y="205" text-anchor="middle" fill="#888" font-size="11">2024</text>
  <!-- VDM -->
  <g transform="translate(40,65)">
    <rect x="0" y="0" width="110" height="70" rx="6" fill="#e8f0fe" stroke="#4285f4" stroke-width="1.5"/>
    <text x="55" y="18" text-anchor="middle" font-weight="bold" fill="#1a73e8" font-size="11">VDM</text>
    <text x="55" y="34" text-anchor="middle" fill="#555" font-size="10">Ho et al. 2022</text>
    <text x="55" y="52" text-anchor="middle" fill="#888" font-size="9">canonical recipe</text>
    <line x1="55" y1="70" x2="55" y2="115" stroke="#999" stroke-width="1"/>
    <circle cx="55" cy="115" r="3" fill="#4285f4"/>
  </g>
  <!-- Imagen Video -->
  <g transform="translate(155,65)">
    <rect x="0" y="0" width="90" height="70" rx="6" fill="#fce8e6" stroke="#ea4335" stroke-width="1.5"/>
    <text x="45" y="18" text-anchor="middle" font-weight="bold" fill="#c5221f" font-size="11">Imagen</text>
    <text x="45" y="30" text-anchor="middle" font-weight="bold" fill="#c5221f" font-size="11">Video</text>
    <text x="45" y="47" text-anchor="middle" fill="#555" font-size="10">Google</text>
    <text x="45" y="60" text-anchor="middle" fill="#888" font-size="9">7-stage cascade</text>
    <line x1="45" y1="70" x2="45" y2="115" stroke="#999" stroke-width="1"/>
    <circle cx="45" cy="115" r="3" fill="#ea4335"/>
  </g>
  <!-- Make-A-Video -->
  <g transform="translate(250,45)">
    <rect x="0" y="0" width="90" height="70" rx="6" fill="#f3e8fd" stroke="#9334e6" stroke-width="1.5"/>
    <text x="45" y="18" text-anchor="middle" font-weight="bold" fill="#7627bb" font-size="11">Make-A-</text>
    <text x="45" y="30" text-anchor="middle" font-weight="bold" fill="#7627bb" font-size="11">Video</text>
    <text x="45" y="47" text-anchor="middle" fill="#555" font-size="10">Meta</text>
    <text x="45" y="60" text-anchor="middle" fill="#888" font-size="9">frozen T2I base</text>
    <line x1="45" y1="70" x2="45" y2="135" stroke="#999" stroke-width="1"/>
    <circle cx="45" cy="135" r="3" fill="#9334e6"/>
  </g>
  <!-- VideoLDM -->
  <g transform="translate(355,65)">
    <rect x="0" y="0" width="100" height="70" rx="6" fill="#fef7e0" stroke="#f9ab00" stroke-width="1.5"/>
    <text x="50" y="18" text-anchor="middle" font-weight="bold" fill="#e37400" font-size="11">Video LDM</text>
    <text x="50" y="34" text-anchor="middle" fill="#555" font-size="10">NVIDIA 2023</text>
    <text x="50" y="52" text-anchor="middle" fill="#888" font-size="9">LDM + temporal</text>
    <line x1="50" y1="70" x2="50" y2="115" stroke="#999" stroke-width="1"/>
    <circle cx="50" cy="115" r="3" fill="#f9ab00"/>
  </g>
  <!-- SVD -->
  <g transform="translate(465,45)">
    <rect x="0" y="0" width="90" height="70" rx="6" fill="#e8f5e9" stroke="#34a853" stroke-width="1.5"/>
    <text x="45" y="18" text-anchor="middle" font-weight="bold" fill="#1b5e20" font-size="11">SVD</text>
    <text x="45" y="34" text-anchor="middle" fill="#555" font-size="10">Stability 2023</text>
    <text x="45" y="52" text-anchor="middle" fill="#888" font-size="9">our demo #1</text>
    <line x1="45" y1="70" x2="45" y2="135" stroke="#999" stroke-width="1"/>
    <circle cx="45" cy="135" r="3" fill="#34a853"/>
  </g>
  <!-- Lumiere -->
  <g transform="translate(565,65)">
    <rect x="0" y="0" width="90" height="70" rx="6" fill="#fce4ec" stroke="#e91e63" stroke-width="1.5"/>
    <text x="45" y="18" text-anchor="middle" font-weight="bold" fill="#880e4f" font-size="11">Lumiere</text>
    <text x="45" y="34" text-anchor="middle" fill="#555" font-size="10">Google 2024</text>
    <text x="45" y="52" text-anchor="middle" fill="#888" font-size="9">Space-Time U-Net</text>
    <line x1="45" y1="70" x2="45" y2="115" stroke="#999" stroke-width="1"/>
    <circle cx="45" cy="115" r="3" fill="#e91e63"/>
  </g>
</svg>
</center>

| Model | Year | Trick |
|---|---|---|
| **VDM** (Ho et al.) | 2022 | Canonical recipe: factorized space-time 3D U-Net |
| **Imagen Video** (Google) | 2022 | Cascade of **7** models: base + 3 spatial SR + 3 temporal SR |
| **Make-A-Video** (Meta) | 2022 | Temporal layers stacked on a **frozen** T2I backbone — no paired video-text needed |
| **Video LDM / Align Your Latents** (NVIDIA) | 2023 | Insert temporal conv + attn into a pretrained image LDM; train only the new layers |
| **Stable Video Diffusion** | 2023 | Same recipe, fine-tune everything; 14–25 frames at 576×1024. Our live-demo baseline. |
| **Lumiere** (Google) | 2024 | **Space-Time U-Net** generating the whole video in one pass (not a cascade) |

All of these stay in the **U-Net + ε-prediction** regime — the image side of the field had not yet made the transformer jump.

> **References**: Ho et al. "Video Diffusion Models" arXiv:2204.03458 · Ho et al. "Imagen Video" 2022 · Singer et al. "Make-A-Video" 2022 · Blattmann et al. "Align Your Latents / Video LDM" 2023 · Blattmann et al. "Stable Video Diffusion" 2023 · Bar-Tal et al. "Lumiere" 2024

### 1.4 The Paradigm Shift — DiT for Video

We saw in Seminar 11 that images went `UNet → DiT → MMDiT → FLUX`. Video follows the same trajectory, about six months behind. The translation is straightforward: reuse the image DiT block, but feed it **spacetime patches** instead of image patches.

**Patchification in 4D**. A latent video $\mathbf{z} \in \mathbb{R}^{T' \times H' \times W' \times C}$ is cut into **tubelets** of shape $p_t \times p_h \times p_w$ (e.g. $1 \times 2 \times 2$ or $2 \times 2 \times 2$):

$$\mathbf{z} \;\xrightarrow{\text{patch embed}}\; \mathbf{s} \in \mathbb{R}^{N \times D},\qquad
N = \frac{T' H' W'}{p_t p_h p_w}$$

<center>
<svg width="700" height="230" xmlns="http://www.w3.org/2000/svg" font-family="Arial, sans-serif" font-size="11">
  <defs><marker id="a4" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#666"/></marker></defs>
  <text x="350" y="18" text-anchor="middle" font-weight="bold" font-size="13" fill="#333">Spacetime-patch pipeline (Sora / CogVideoX / HunyuanVideo / Wan)</text>
  <!-- Pixel video -->
  <g transform="translate(20,50)">
    <rect x="0"  y="0"  width="70" height="50" rx="2" fill="#e8f0fe" stroke="#4285f4"/>
    <rect x="7"  y="7"  width="70" height="50" rx="2" fill="#e8f0fe" stroke="#4285f4"/>
    <rect x="14" y="14" width="70" height="50" rx="2" fill="#e8f0fe" stroke="#4285f4"/>
    <rect x="21" y="21" width="70" height="50" rx="2" fill="#e8f0fe" stroke="#4285f4"/>
    <text x="55" y="48" text-anchor="middle" fill="#1a73e8" font-weight="bold" font-size="11">T×H×W×3</text>
    <text x="50" y="95" text-anchor="middle" fill="#888" font-size="10">pixel video</text>
  </g>
  <line x1="125" y1="75" x2="170" y2="75" stroke="#666" stroke-width="1.5" marker-end="url(#a4)"/>
  <text x="145" y="65" text-anchor="middle" fill="#888" font-size="10">3D causal</text>
  <text x="145" y="95" text-anchor="middle" fill="#888" font-size="10">VAE encode</text>
  <!-- Latent video -->
  <g transform="translate(175,55)">
    <rect x="0"  y="0"  width="60" height="40" rx="2" fill="#ede7f6" stroke="#7c4dff"/>
    <rect x="6"  y="6"  width="60" height="40" rx="2" fill="#ede7f6" stroke="#7c4dff"/>
    <rect x="12" y="12" width="60" height="40" rx="2" fill="#ede7f6" stroke="#7c4dff"/>
    <text x="45" y="38" text-anchor="middle" fill="#4a148c" font-weight="bold" font-size="11">T'×H'×W'×C</text>
    <text x="40" y="85" text-anchor="middle" fill="#888" font-size="10">latent video</text>
  </g>
  <line x1="260" y1="75" x2="305" y2="75" stroke="#666" stroke-width="1.5" marker-end="url(#a4)"/>
  <text x="282" y="65" text-anchor="middle" fill="#888" font-size="10">patchify</text>
  <text x="282" y="95" text-anchor="middle" fill="#888" font-size="10">pₜ×p_h×p_w</text>
  <!-- Token sequence -->
  <g transform="translate(310,55)">
    <rect x="0" y="0" width="14" height="20" rx="2" fill="#fce4ec" stroke="#e91e63"/>
    <rect x="18" y="0" width="14" height="20" rx="2" fill="#fce4ec" stroke="#e91e63"/>
    <rect x="36" y="0" width="14" height="20" rx="2" fill="#fce4ec" stroke="#e91e63"/>
    <rect x="54" y="0" width="14" height="20" rx="2" fill="#fce4ec" stroke="#e91e63"/>
    <rect x="72" y="0" width="14" height="20" rx="2" fill="#fce4ec" stroke="#e91e63"/>
    <text x="95" y="14" fill="#880e4f" font-size="11">· · ·</text>
    <rect x="110" y="0" width="14" height="20" rx="2" fill="#fce4ec" stroke="#e91e63"/>
    <rect x="128" y="0" width="14" height="20" rx="2" fill="#fce4ec" stroke="#e91e63"/>
    <text x="75" y="38" text-anchor="middle" fill="#880e4f" font-weight="bold" font-size="11">N = T'H'W' / (pₜp_hp_w)</text>
    <text x="75" y="54" text-anchor="middle" fill="#555" font-size="10">spacetime tokens, 3D position</text>
    <text x="75" y="85" text-anchor="middle" fill="#888" font-size="10">token sequence</text>
  </g>
  <line x1="470" y1="75" x2="515" y2="75" stroke="#666" stroke-width="1.5" marker-end="url(#a4)"/>
  <!-- DiT -->
  <g transform="translate(520,50)">
    <rect x="0" y="0" width="135" height="55" rx="6" fill="#fef7e0" stroke="#f9ab00" stroke-width="2"/>
    <text x="67" y="24" text-anchor="middle" font-weight="bold" fill="#e37400" font-size="12">DiT blocks</text>
    <text x="67" y="42" text-anchor="middle" fill="#555" font-size="10">variable shape in</text>
    <text x="67" y="85" text-anchor="middle" fill="#888" font-size="10">transformer</text>
  </g>
  <!-- Caption under -->
  <text x="350" y="175" text-anchor="middle" fill="#444" font-size="11" font-style="italic">
    Consequence: transformer sees only a flat sequence → variable resolution, duration, aspect ratio are &quot;free&quot;.
  </text>
  <rect x="30" y="190" width="640" height="30" rx="6" fill="#f3f4f6" stroke="#bbb"/>
  <text x="350" y="210" text-anchor="middle" fill="#333" font-size="11">
    e.g. N=1024 can be <tspan fill="#1a73e8">8×32×32</tspan> or <tspan fill="#1a73e8">16×16×32</tspan> or <tspan fill="#1a73e8">32×8×32</tspan> — same model weights.
  </text>
</svg>
</center>

This single change has a big consequence: because the transformer only sees a **sequence of tokens**, the model becomes **resolution- and duration-agnostic** — an `N=1024` sequence can be $8\times32\times32$ just as easily as $16\times16\times32$. Sora's variable-resolution, variable-duration inference is exactly this property being exploited.

### 1.5 The 3D Causal VAE — The New Primitive

The 2D image VAE from SD is no longer sufficient: we need to compress both space and time, and we want the encoder to be **streaming-friendly** so that we can extend / stitch long videos.

**Building block — CausalConv3D**. A 3D convolution with padding applied **only in the past direction** along the time axis. Every output frame depends only on current + previous input frames, never on future frames — the causal property.

<center>
<svg width="700" height="270" xmlns="http://www.w3.org/2000/svg" font-family="Arial, sans-serif" font-size="11">
  <defs><marker id="a5" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#666"/></marker></defs>
  <text x="350" y="18" text-anchor="middle" font-weight="bold" font-size="13" fill="#333">3D causal VAE — 8×8 spatial · 4× temporal</text>
  <!-- Input box -->
  <g transform="translate(25,55)">
    <rect x="0"  y="0"  width="140" height="80" rx="3" fill="#e8f0fe" stroke="#4285f4" stroke-width="2"/>
    <text x="70"  y="22" text-anchor="middle" fill="#1a73e8" font-weight="bold" font-size="12">Pixel video</text>
    <text x="70"  y="40" text-anchor="middle" fill="#333" font-size="11">129 × 720 × 1280 × 3</text>
    <text x="70"  y="58" text-anchor="middle" fill="#888" font-size="10">≈ 3.6 · 10⁸ elements</text>
    <text x="70"  y="75" text-anchor="middle" fill="#888" font-size="10">(5 s at ~26 fps · 720p)</text>
  </g>
  <line x1="170" y1="95" x2="230" y2="95" stroke="#666" stroke-width="1.5" marker-end="url(#a5)"/>
  <text x="200" y="85" text-anchor="middle" fill="#888" font-size="10">encode</text>
  <!-- VAE block -->
  <g transform="translate(235,35)">
    <rect x="0" y="0" width="220" height="120" rx="8" fill="#fff9f0" stroke="#f9ab00" stroke-width="2"/>
    <text x="110" y="22" text-anchor="middle" font-weight="bold" fill="#e37400" font-size="12">3D Causal VAE</text>
    <!-- subblock: CausalConv3D -->
    <rect x="15" y="32" width="190" height="24" rx="4" fill="#fef7e0" stroke="#f9ab00"/>
    <text x="110" y="48" text-anchor="middle" fill="#e37400" font-size="10" font-weight="bold">CausalConv3D · downsample 8×8×4</text>
    <!-- subblock: anti-flicker loss -->
    <rect x="15" y="62" width="190" height="24" rx="4" fill="#fde7f3" stroke="#e91e63"/>
    <text x="110" y="78" text-anchor="middle" fill="#880e4f" font-size="10" font-weight="bold">+ temporal anti-flicker loss</text>
    <!-- subblock: tiling -->
    <rect x="15" y="92" width="190" height="22" rx="4" fill="#e8f5e9" stroke="#34a853"/>
    <text x="110" y="107" text-anchor="middle" fill="#1b5e20" font-size="10" font-weight="bold">+ spatial/temporal tiling</text>
  </g>
  <line x1="460" y1="95" x2="515" y2="95" stroke="#666" stroke-width="1.5" marker-end="url(#a5)"/>
  <!-- Latent box -->
  <g transform="translate(520,55)">
    <rect x="0"  y="0"  width="150" height="80" rx="3" fill="#ede7f6" stroke="#7c4dff" stroke-width="2"/>
    <text x="75"  y="22" text-anchor="middle" fill="#4a148c" font-weight="bold" font-size="12">Latent video</text>
    <text x="75"  y="40" text-anchor="middle" fill="#333" font-size="11">33 × 90 × 160 × 16</text>
    <text x="75"  y="58" text-anchor="middle" fill="#888" font-size="10">≈ 1.9 · 10⁶ elements</text>
    <text x="75"  y="75" text-anchor="middle" fill="#34a853" font-weight="bold" font-size="11">≈ 185× smaller</text>
  </g>
  <!-- Causal arrow diagram -->
  <g transform="translate(120,185)">
    <text x="230" y="-4" text-anchor="middle" fill="#444" font-weight="bold" font-size="11">Causal convolution — time axis</text>
    <circle cx="30"  cy="20" r="6" fill="#bbd"/>
    <circle cx="80"  cy="20" r="6" fill="#bbd"/>
    <circle cx="130" cy="20" r="6" fill="#cfe2ff" stroke="#1a73e8"/>
    <circle cx="180" cy="20" r="6" fill="#cfe2ff" stroke="#1a73e8"/>
    <circle cx="230" cy="20" r="6" fill="#fde7f3" stroke="#e91e63"/>
    <circle cx="280" cy="20" r="6" fill="#e8f0fe" stroke="#4285f4"/>
    <circle cx="330" cy="20" r="6" fill="#e8f0fe" stroke="#4285f4"/>
    <circle cx="380" cy="20" r="6" fill="#e8f0fe" stroke="#4285f4"/>
    <circle cx="430" cy="20" r="6" fill="#e8f0fe" stroke="#4285f4"/>
    <text x="30"  y="40" text-anchor="middle" fill="#888" font-size="9">t−3</text>
    <text x="80"  y="40" text-anchor="middle" fill="#888" font-size="9">t−2</text>
    <text x="130" y="40" text-anchor="middle" fill="#888" font-size="9">t−1</text>
    <text x="180" y="40" text-anchor="middle" fill="#1a73e8" font-size="9" font-weight="bold">t</text>
    <text x="230" y="40" text-anchor="middle" fill="#c5221f" font-size="9" font-weight="bold">t+1</text>
    <text x="280" y="40" text-anchor="middle" fill="#888" font-size="9">t+2</text>
    <text x="330" y="40" text-anchor="middle" fill="#888" font-size="9">t+3</text>
    <!-- Kernel for output at t+1 -->
    <path d="M 130 13 Q 175 -4 225 13" fill="none" stroke="#e91e63" stroke-width="1" opacity="0.8"/>
    <path d="M 180 13 Q 200 -2 225 13" fill="none" stroke="#e91e63" stroke-width="1" opacity="0.8"/>
    <text x="45" y="65" fill="#888" font-size="9">past</text>
    <text x="230" y="65" text-anchor="middle" fill="#c5221f" font-size="10" font-weight="bold">output at t+1 reads only ≤ t+1</text>
    <text x="400" y="65" fill="#888" font-size="9">future — unused</text>
  </g>
</svg>
</center>

**Typical compression**: `8× spatial × 8× spatial × 4× temporal`. A 720×1280×129 video (3 channels) compresses as:
$$(T, H, W, C) = (129, 720, 1280, 3) \;\xrightarrow{\text{VAE}}\; (T', H', W', C_z) = (33, 90, 160, 16)$$

which is $\approx 185 \times$ fewer elements — enough to make a 3D transformer tractable.

**Latent shape in general**:
$$\mathbf{z} \in \mathbb{R}^{T/4 \,\times\, H/8 \,\times\, W/8 \,\times\, C_z}, \qquad C_z \in \{4, 16\}$$

**Tiling for long videos**. For videos longer than the VAE was trained on, encode and decode in **overlapping spatial + temporal tiles** and blend at the seams. Without tiling the 3D VAE alone exhausts VRAM long before the transformer does.

Anti-flicker loss. Modern 3D VAEs (CogVideoX, HunyuanVideo) are trained with explicit **temporal smoothness** losses on the reconstruction — this is the first line of defence against per-frame flickering.

### 1.6 3D RoPE — Positional Encoding for Spacetime

Positional encoding must generalise across resolutions, aspect ratios, and video lengths. Absolute / learned PE is a bad fit — RoPE (Seminar 11) solves this for images. The extension to video is straightforward: split the head dimension and rotate each slice independently per axis.

**Construction**. Decompose the per-head embedding dimension:
$$d = d_t + d_h + d_w$$
and for a token at position $(m_t, m_h, m_w)$ apply **three independent RoPE rotations**:
$$\text{RoPE}_t(m_t) \;\oplus\; \text{RoPE}_h(m_h) \;\oplus\; \text{RoPE}_w(m_w)$$

<center>
<svg width="700" height="200" xmlns="http://www.w3.org/2000/svg" font-family="Arial, sans-serif" font-size="11">
  <text x="350" y="18" text-anchor="middle" font-weight="bold" font-size="13" fill="#333">Per-head dimension split — one RoPE family per axis</text>
  <!-- Full head vector -->
  <g transform="translate(30,45)">
    <rect x="0" y="0" width="640" height="44" rx="4" fill="#f3f4f6" stroke="#bbb"/>
    <text x="320" y="-10" text-anchor="middle" fill="#333" font-weight="bold" font-size="11">head embedding (d = 128)</text>
    <!-- time slice -->
    <rect x="0" y="0" width="80" height="44" rx="4" fill="#e8f0fe" stroke="#4285f4" stroke-width="1.5"/>
    <text x="40" y="20" text-anchor="middle" fill="#1a73e8" font-weight="bold" font-size="11">dₜ = 16</text>
    <text x="40" y="36" text-anchor="middle" fill="#555" font-size="10">RoPEₜ(mₜ)</text>
    <!-- height slice -->
    <rect x="80" y="0" width="280" height="44" rx="4" fill="#fef7e0" stroke="#f9ab00" stroke-width="1.5"/>
    <text x="220" y="20" text-anchor="middle" fill="#e37400" font-weight="bold" font-size="11">d_h = 56</text>
    <text x="220" y="36" text-anchor="middle" fill="#555" font-size="10">RoPE_h(m_h)</text>
    <!-- width slice -->
    <rect x="360" y="0" width="280" height="44" rx="4" fill="#fce4ec" stroke="#e91e63" stroke-width="1.5"/>
    <text x="500" y="20" text-anchor="middle" fill="#880e4f" font-weight="bold" font-size="11">d_w = 56</text>
    <text x="500" y="36" text-anchor="middle" fill="#555" font-size="10">RoPE_w(m_w)</text>
  </g>
  <!-- rotations illustration -->
  <g transform="translate(70,125)">
    <text x="-10" y="5" fill="#888" font-size="10">θ(m) =</text>
    <circle cx="60" cy="0" r="18" fill="none" stroke="#4285f4" stroke-width="1.3"/>
    <line x1="60" y1="0" x2="76" y2="-8" stroke="#4285f4" stroke-width="2"/>
    <text x="60" y="36" text-anchor="middle" fill="#1a73e8" font-size="10">time angle · mₜ</text>
    <circle cx="200" cy="0" r="18" fill="none" stroke="#f9ab00" stroke-width="1.3"/>
    <line x1="200" y1="0" x2="212" y2="-14" stroke="#f9ab00" stroke-width="2"/>
    <text x="200" y="36" text-anchor="middle" fill="#e37400" font-size="10">height angle · m_h</text>
    <circle cx="360" cy="0" r="18" fill="none" stroke="#e91e63" stroke-width="1.3"/>
    <line x1="360" y1="0" x2="378" y2="-4" stroke="#e91e63" stroke-width="2"/>
    <text x="360" y="36" text-anchor="middle" fill="#880e4f" font-size="10">width angle · m_w</text>
    <text x="430" y="5" fill="#333" font-size="11">→</text>
    <rect x="460" y="-18" width="140" height="36" rx="4" fill="#e8f5e9" stroke="#34a853"/>
    <text x="530" y="-2" text-anchor="middle" fill="#1b5e20" font-weight="bold" font-size="11">relative distances</text>
    <text x="530" y="13" text-anchor="middle" fill="#555" font-size="10">along each axis, independently</text>
  </g>
</svg>
</center>

Each sub-slice encodes **relative** distance along one axis. The whole scheme is permutation-equivariant in a useful way: rotating the video in time or shifting it in space leaves attention scores invariant up to the expected relative offset.

[**Accent**] 3D RoPE is the default in CogVideoX, HunyuanVideo, LTX, and Wan. It is the single encoding change that makes generating at multiple resolutions/lengths work reliably.

> **Reference**: VideoRoPE arXiv:2502.05173

---
*With factorized / joint space-time attention, 3D causal VAE, and 3D RoPE as our primitives, we can now look at concrete modern architectures.*

## 2. Architecture Overview — Sora, Kling, CogVideoX, HunyuanVideo, Mochi, LTX, Wan, Veo

We walk the 2024–2025 landscape. For each model the question is always the same three-slot template: **what backbone · what VAE · what text encoder · what objective**. The answers tell a very consistent story.

### 2.1 Sora — OpenAI (Feb 2024, closed)

The model that started the modern video race. Only a technical report was released, but the key design choices are documented.

- **Backbone**: DiT on spacetime patches (no factorization — full joint attention).
- **VAE**: a learned **spatiotemporal** autoencoder trained from scratch — not the SD image VAE.
- **Positional encoding**: 3D absolute PE over $(t, h, w)$ — allows **variable resolution, duration, and aspect ratio** as direct inputs.
- **Parameter count**: undisclosed; community estimates $\sim 3{-}10$B.
- **Training trick**: "DALL-E 3 re-captioning" — a dense captioner re-labels every training video, massively boosting prompt adherence.

<center>
<svg width="700" height="230" xmlns="http://www.w3.org/2000/svg" font-family="Arial, sans-serif" font-size="11">
  <defs><marker id="a6" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#666"/></marker></defs>
  <text x="350" y="18" text-anchor="middle" font-weight="bold" font-size="13" fill="#333">Sora — DiT on Spacetime Patches</text>
  <!-- video input -->
  <g transform="translate(15,55)">
    <rect x="0"  y="0"  width="90" height="60" rx="3" fill="#e8f0fe" stroke="#4285f4"/>
    <rect x="7"  y="7"  width="90" height="60" rx="3" fill="#e8f0fe" stroke="#4285f4"/>
    <rect x="14" y="14" width="90" height="60" rx="3" fill="#e8f0fe" stroke="#4285f4"/>
    <text x="65" y="54" text-anchor="middle" fill="#1a73e8" font-weight="bold" font-size="11">video</text>
    <text x="52" y="95" text-anchor="middle" fill="#888" font-size="10">variable T, H, W</text>
  </g>
  <line x1="125" y1="90" x2="165" y2="90" stroke="#666" stroke-width="1.5" marker-end="url(#a6)"/>
  <!-- spatiotemporal VAE -->
  <g transform="translate(170,65)">
    <rect x="0" y="0" width="130" height="45" rx="6" fill="#fff9f0" stroke="#f9ab00" stroke-width="2"/>
    <text x="65" y="20" text-anchor="middle" fill="#e37400" font-weight="bold" font-size="11">Spatiotemporal</text>
    <text x="65" y="34" text-anchor="middle" fill="#e37400" font-weight="bold" font-size="11">VAE</text>
    <text x="65" y="62" text-anchor="middle" fill="#888" font-size="10">latent (T', H', W', C)</text>
  </g>
  <line x1="305" y1="90" x2="345" y2="90" stroke="#666" stroke-width="1.5" marker-end="url(#a6)"/>
  <!-- patchify -->
  <g transform="translate(350,65)">
    <rect x="0" y="0" width="110" height="45" rx="6" fill="#ede7f6" stroke="#7c4dff" stroke-width="2"/>
    <text x="55" y="20" text-anchor="middle" fill="#4a148c" font-weight="bold" font-size="11">patchify</text>
    <text x="55" y="34" text-anchor="middle" fill="#555" font-size="10">tubelets 2×2×2</text>
    <text x="55" y="62" text-anchor="middle" fill="#888" font-size="10">N tokens + 3D PE</text>
  </g>
  <line x1="465" y1="90" x2="505" y2="90" stroke="#666" stroke-width="1.5" marker-end="url(#a6)"/>
  <!-- DiT -->
  <g transform="translate(510,55)">
    <rect x="0" y="0" width="170" height="65" rx="8" fill="#fef7e0" stroke="#f9ab00" stroke-width="2.5"/>
    <text x="85" y="24" text-anchor="middle" font-weight="bold" fill="#e37400" font-size="13">DiT blocks</text>
    <text x="85" y="42" text-anchor="middle" fill="#555" font-size="10">full joint attention</text>
    <text x="85" y="57" text-anchor="middle" fill="#555" font-size="10">~3–10B params</text>
  </g>
  <!-- output chain -->
  <line x1="595" y1="120" x2="595" y2="148" stroke="#666" stroke-width="1.5" marker-end="url(#a6)"/>
  <g transform="translate(510,148)">
    <rect x="0" y="0" width="170" height="30" rx="4" fill="#e6f4ea" stroke="#34a853"/>
    <text x="85" y="20" text-anchor="middle" fill="#1b5e20" font-weight="bold" font-size="11">Spatiotemporal VAE decode</text>
  </g>
  <line x1="510" y1="163" x2="470" y2="163" stroke="#666" stroke-width="1.5" marker-end="url(#a6)"/>
  <rect x="380" y="148" width="90" height="30" rx="4" fill="#e8f0fe" stroke="#4285f4"/>
  <text x="425" y="168" text-anchor="middle" fill="#1a73e8" font-weight="bold" font-size="11">video out</text>
  <!-- Trick badge -->
  <rect x="40" y="170" width="250" height="42" rx="8" fill="#fce4ec" stroke="#e91e63" stroke-width="1.5"/>
  <text x="165" y="188" text-anchor="middle" fill="#880e4f" font-weight="bold" font-size="11">+ DALL·E-3 re-captioning</text>
  <text x="165" y="204" text-anchor="middle" fill="#555" font-size="10">dense captioner relabels every training clip</text>
</svg>
</center>

[**Accent**] Takeaway: "DiT on spacetime patches with variable input shape" is the modern recipe. Every other DiT-based video model below is a variation of Sora.

> **Reference**: Brooks et al. "Video generation models as world simulators" OpenAI tech report, Feb 2024

### 2.2 Kling — Kuaishou (June 2024, closed API)

- **Backbone**: DiT with **full spatiotemporal self-attention** (no factorization).
- **VAE**: in-house 3D VAE.
- **Notable spec**: up to **2-minute clips at 1080p, 30 fps** — the longest among closed models.
- No architectural paper: the design is reconstructed from product blog posts and interviews.

Kling's existence shows that **full 3D attention** is feasible at scale given enough compute; the open community responded with sparse / factorized attention (see §5.2).

### 2.3 CogVideoX — Tsinghua / Zhipu (Aug 2024, ICLR 2025, OPEN)

Our **main case study**: the best open-source DiT-based video model with full architectural details in the paper.

- **Variants**: `CogVideoX-2B`, `CogVideoX-5B`, `CogVideoX1.5-5B`, plus I2V variants.
- **Output**: 6–10 s videos at 16 fps, up to 768×1360 (5B) / 720×480 (2B).
- **Text encoder**: T5-v1.1-xxl (4096-dim) — same T5 branch as SD 3.

<center>
<svg width="720" height="240" xmlns="http://www.w3.org/2000/svg" font-family="Arial, sans-serif" font-size="11">
  <defs><marker id="aC" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#666"/></marker></defs>
  <text x="360" y="18" text-anchor="middle" font-weight="bold" font-size="13" fill="#333">CogVideoX full pipeline</text>
  <!-- Text encoder -->
  <rect x="20" y="55" width="120" height="40" rx="6" fill="#fce8e6" stroke="#ea4335" stroke-width="2"/>
  <text x="80" y="75" text-anchor="middle" fill="#c5221f" font-weight="bold">T5-v1.1-xxl</text>
  <text x="80" y="90" text-anchor="middle" fill="#555" font-size="10">4096-dim</text>
  <!-- Noise -->
  <rect x="20" y="130" width="120" height="40" rx="6" fill="#f3e8fd" stroke="#9334e6" stroke-width="2"/>
  <text x="80" y="150" text-anchor="middle" fill="#7627bb" font-weight="bold">noise latent</text>
  <text x="80" y="165" text-anchor="middle" fill="#555" font-size="10">T'×H'×W'×16</text>
  <!-- Expert MMDiT -->
  <g transform="translate(170,45)">
    <rect x="0" y="0" width="220" height="140" rx="10" fill="#fff9f0" stroke="#f9ab00" stroke-width="2.5"/>
    <text x="110" y="22" text-anchor="middle" fill="#e37400" font-weight="bold" font-size="13">Expert MMDiT</text>
    <text x="110" y="40" text-anchor="middle" fill="#888" font-size="10">2B / 5B params</text>
    <!-- Stacked blocks -->
    <rect x="30" y="55" width="160" height="18" rx="4" fill="#fef7e0" stroke="#f9ab00"/>
    <text x="110" y="68" text-anchor="middle" fill="#e37400" font-size="10">joint self-attn</text>
    <rect x="30" y="78" width="160" height="18" rx="4" fill="#fde7f3" stroke="#e91e63"/>
    <text x="110" y="91" text-anchor="middle" fill="#880e4f" font-size="10">AdaLN_video · AdaLN_text</text>
    <rect x="30" y="101" width="160" height="18" rx="4" fill="#e8f0fe" stroke="#4285f4"/>
    <text x="110" y="114" text-anchor="middle" fill="#1a73e8" font-size="10">FFN_v · FFN_t</text>
    <text x="110" y="132" text-anchor="middle" fill="#555" font-size="10">× N layers · 3D RoPE</text>
  </g>
  <!-- arrows from T5 and noise to MMDiT -->
  <line x1="140" y1="75"  x2="170" y2="85" stroke="#ea4335" stroke-width="1.5" marker-end="url(#aC)"/>
  <line x1="140" y1="150" x2="170" y2="140" stroke="#9334e6" stroke-width="1.5" marker-end="url(#aC)"/>
  <!-- Timestep modulation -->
  <rect x="200" y="195" width="160" height="28" rx="5" fill="#fff3e0" stroke="#ff9800" stroke-width="1.5"/>
  <text x="280" y="213" text-anchor="middle" fill="#e65100" font-size="10" font-weight="bold">c = t_emb → (γ_v,β_v) (γ_t,β_t)</text>
  <line x1="280" y1="195" x2="280" y2="185" stroke="#ff9800" stroke-width="1.2" stroke-dasharray="3,2" marker-end="url(#aC)"/>
  <!-- 3D causal VAE -->
  <line x1="395" y1="115" x2="440" y2="115" stroke="#666" stroke-width="1.5" marker-end="url(#aC)"/>
  <g transform="translate(445,85)">
    <rect x="0" y="0" width="130" height="60" rx="8" fill="#e8f5e9" stroke="#34a853" stroke-width="2"/>
    <text x="65" y="20" text-anchor="middle" fill="#1b5e20" font-weight="bold" font-size="12">3D Causal VAE</text>
    <text x="65" y="36" text-anchor="middle" fill="#555" font-size="10">decode 8×8 spatial</text>
    <text x="65" y="50" text-anchor="middle" fill="#555" font-size="10">4× temporal</text>
  </g>
  <line x1="580" y1="115" x2="620" y2="115" stroke="#666" stroke-width="1.5" marker-end="url(#aC)"/>
  <!-- Video out -->
  <g transform="translate(625,85)">
    <rect x="0" y="0" width="70" height="60" rx="3" fill="#e8f0fe" stroke="#4285f4"/>
    <text x="35" y="35" text-anchor="middle" fill="#1a73e8" font-weight="bold" font-size="11">video</text>
    <text x="35" y="51" text-anchor="middle" fill="#555" font-size="10">6–10 s, 16 fps</text>
  </g>
</svg>
</center>

**Three core innovations** — each gets its own slide below.

> **Reference**: Yang et al. "CogVideoX: Text-to-Video Diffusion Models with an Expert Transformer", arXiv:2408.06072, ICLR 2025. HF IDs: `zai-org/CogVideoX-{2b,5b,5b-I2V}`.

#### 2.3.1 CogVideoX Innovation 1 — 3D Causal VAE with $8 \times 8 \times 4$ Compression

- CausalConv3D stack, spatial downsample $\times 8$ per side, temporal downsample $\times 4$.
- Trained with **explicit anti-flicker reconstruction losses** (consecutive-frame L1 and feature-matching) — this is where "the cat's fur does not shimmer" comes from.
- 16 latent channels (more than SD's 4) → tighter reconstruction, easier for the transformer.
- Supports **tiling** at inference for longer clips than the training length.

#### 2.3.2 CogVideoX Innovation 2 — Expert MMDiT + Expert-AdaLN (the key novelty)

Recall MMDiT from Seminar 11: text and video tokens are concatenated and passed through **joint self-attention**, with separate Q/K/V/FFN per modality. CogVideoX keeps all of that. The new piece is that the **AdaLN timestep modulation is now also per-modality**:

$$\text{AdaLN}_{\text{video}}(\mathbf{h}_v,\, \mathbf{c}) = \gamma_v(\mathbf{c})\, \text{LN}(\mathbf{h}_v) + \beta_v(\mathbf{c})$$
$$\text{AdaLN}_{\text{text}}(\mathbf{h}_t,\, \mathbf{c}) = \gamma_t(\mathbf{c})\, \text{LN}(\mathbf{h}_t) + \beta_t(\mathbf{c})$$

with separate MLPs producing $(\gamma_v, \beta_v)$ and $(\gamma_t, \beta_t)$ from the shared conditioning vector $\mathbf{c}$ (timestep embedding).

[**Accent**] Contrast with SD 3: in SD3-MMDiT a **single** AdaLN modulates both streams. Here each modality has its **own "expert" modulation** — hence "Expert MMDiT". Since video and text statistics (scale, sparsity, temporal structure) are very different, this per-modality conditioning is measurably better than shared AdaLN in the ablations.

<center>
<svg width="700" height="440" xmlns="http://www.w3.org/2000/svg" font-family="Arial, sans-serif" font-size="11">
  <defs><marker id="a7" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#666"/></marker></defs>
  <rect x="1" y="1" width="698" height="438" rx="12" fill="none" stroke="#ccc" stroke-width="1.5" stroke-dasharray="6,3"/>
  <text x="350" y="22" text-anchor="middle" font-weight="bold" font-size="14" fill="#333">Expert MMDiT block (CogVideoX)</text>
  <!-- Video input -->
  <rect x="30"  y="45" width="180" height="40" rx="6" fill="#e8f0fe" stroke="#4285f4" stroke-width="2"/>
  <text x="120" y="70" text-anchor="middle" fill="#1a73e8" font-weight="bold" font-size="12">video tokens z_v</text>
  <!-- Text input -->
  <rect x="490" y="45" width="180" height="40" rx="6" fill="#fce8e6" stroke="#ea4335" stroke-width="2"/>
  <text x="580" y="70" text-anchor="middle" fill="#c5221f" font-weight="bold" font-size="12">text tokens z_t</text>
  <!-- Timestep c -->
  <rect x="300" y="45" width="100" height="40" rx="6" fill="#f3e8fd" stroke="#9334e6" stroke-width="1.5"/>
  <text x="350" y="63" text-anchor="middle" fill="#7627bb" font-weight="bold" font-size="11">c = t_emb</text>
  <text x="350" y="78" text-anchor="middle" fill="#888" font-size="10">shared conditioning</text>
  <!-- AdaLN video expert -->
  <line x1="120" y1="85" x2="120" y2="110" stroke="#666" stroke-width="1.5" marker-end="url(#a7)"/>
  <rect x="30" y="110" width="180" height="40" rx="6" fill="#fff3e0" stroke="#ff9800" stroke-width="2"/>
  <text x="120" y="128" text-anchor="middle" fill="#e65100" font-weight="bold" font-size="12">AdaLN_video</text>
  <text x="120" y="143" text-anchor="middle" fill="#555" font-size="10">γ_v(c), β_v(c) — MLP_v</text>
  <!-- AdaLN text expert -->
  <line x1="580" y1="85" x2="580" y2="110" stroke="#666" stroke-width="1.5" marker-end="url(#a7)"/>
  <rect x="490" y="110" width="180" height="40" rx="6" fill="#fff3e0" stroke="#ff9800" stroke-width="2"/>
  <text x="580" y="128" text-anchor="middle" fill="#e65100" font-weight="bold" font-size="12">AdaLN_text</text>
  <text x="580" y="143" text-anchor="middle" fill="#555" font-size="10">γ_t(c), β_t(c) — MLP_t</text>
  <!-- c arrows into each AdaLN -->
  <line x1="305" y1="85" x2="210" y2="125" stroke="#9334e6" stroke-width="1.2" stroke-dasharray="4,2" marker-end="url(#a7)"/>
  <line x1="395" y1="85" x2="490" y2="125" stroke="#9334e6" stroke-width="1.2" stroke-dasharray="4,2" marker-end="url(#a7)"/>
  <!-- Q/K/V -->
  <line x1="120" y1="150" x2="120" y2="178" stroke="#666" stroke-width="1.5" marker-end="url(#a7)"/>
  <rect x="30" y="178" width="180" height="30" rx="5" fill="#fce4ec" stroke="#e91e63" stroke-width="1.5"/>
  <text x="120" y="198" text-anchor="middle" fill="#880e4f" font-size="11" font-weight="bold">Q_v, K_v, V_v</text>
  <line x1="580" y1="150" x2="580" y2="178" stroke="#666" stroke-width="1.5" marker-end="url(#a7)"/>
  <rect x="490" y="178" width="180" height="30" rx="5" fill="#fce4ec" stroke="#e91e63" stroke-width="1.5"/>
  <text x="580" y="198" text-anchor="middle" fill="#880e4f" font-size="11" font-weight="bold">Q_t, K_t, V_t</text>
  <!-- Arrows into joint attention -->
  <line x1="180" y1="208" x2="295" y2="245" stroke="#4285f4" stroke-width="2" marker-end="url(#a7)"/>
  <line x1="520" y1="208" x2="405" y2="245" stroke="#ea4335" stroke-width="2" marker-end="url(#a7)"/>
  <!-- Joint attention -->
  <rect x="240" y="245" width="220" height="50" rx="8" fill="#e8f5e9" stroke="#34a853" stroke-width="2.5"/>
  <text x="350" y="266" text-anchor="middle" fill="#1b5e20" font-weight="bold" font-size="13">Joint Self-Attention</text>
  <text x="350" y="283" text-anchor="middle" fill="#555" font-size="11">bidirectional video ↔ text</text>
  <!-- split out -->
  <line x1="295" y1="295" x2="180" y2="325" stroke="#4285f4" stroke-width="2" marker-end="url(#a7)"/>
  <line x1="405" y1="295" x2="520" y2="325" stroke="#ea4335" stroke-width="2" marker-end="url(#a7)"/>
  <!-- FFN -->
  <rect x="30" y="325" width="180" height="36" rx="6" fill="#e8f0fe" stroke="#4285f4" stroke-width="2"/>
  <text x="120" y="348" text-anchor="middle" fill="#1a73e8" font-weight="bold" font-size="12">FFN_video</text>
  <rect x="490" y="325" width="180" height="36" rx="6" fill="#fce8e6" stroke="#ea4335" stroke-width="2"/>
  <text x="580" y="348" text-anchor="middle" fill="#c5221f" font-weight="bold" font-size="12">FFN_text</text>
  <!-- Output labels -->
  <line x1="120" y1="361" x2="120" y2="390" stroke="#666" stroke-width="1.5" marker-end="url(#a7)"/>
  <line x1="580" y1="361" x2="580" y2="390" stroke="#666" stroke-width="1.5" marker-end="url(#a7)"/>
  <text x="120" y="410" text-anchor="middle" fill="#1a73e8" font-weight="bold" font-size="12">z_v'</text>
  <text x="580" y="410" text-anchor="middle" fill="#c5221f" font-weight="bold" font-size="12">z_t'</text>
  <!-- Expert badge -->
  <rect x="260" y="375" width="180" height="45" rx="8" fill="#fff9f0" stroke="#f9ab00" stroke-width="1.5"/>
  <text x="350" y="393" text-anchor="middle" fill="#e37400" font-weight="bold" font-size="11">SD3-MMDiT had ONE AdaLN</text>
  <text x="350" y="410" text-anchor="middle" fill="#555" font-size="11">CogVideoX → per-modality experts</text>
</svg>
</center>

The diagram above emphasises the key difference from SD3-MMDiT: in SD3 a single AdaLN produces one set of $(\gamma, \beta)$ shared across both streams; CogVideoX routes the timestep through two independent MLPs, one per modality. The rest of the block — separate Q/K/V projections, joint self-attention, separate FFNs — is the same MMDiT structure.

#### 2.3.3 CogVideoX Innovation 3 — 3D RoPE

Same 3D RoPE as §1.6 — independent rotary embeddings on $(t, h, w)$. In CogVideoX ablations this converges **substantially faster** than sinusoidal absolute PE and generalises better to unseen resolutions / lengths.

**Training recipe**: progressive low-res → high-res, multi-resolution frame packs, classifier-free guidance on the text prompt, Rectified-Flow / v-prediction objective.

### 2.4 HunyuanVideo — Tencent (Dec 2024, OPEN, 13B)

Too big to run on T4 (needs $\geq$14 GB in FP8 plus an 8B text encoder), but architecturally important: it brings the **FLUX design** to video.

Three components:

1. **3D causal VAE**: CausalConv3D, $8\times$ spatial, $4\times$ temporal, 16 channels — the same template as CogVideoX.

2. **MLLM text encoder** — `xtuner/llava-llama-3-8b-v1_1` decoder-only LLM + a bidirectional **token refiner**. This **replaces T5/CLIP entirely**. Claimed to beat both on image-text alignment; the decoder-only LLM gives richer instruction-following semantics.

3. **Dual-stream → single-stream DiT** (exactly the FLUX pattern from Seminar 11):

<center>
<svg width="700" height="300" xmlns="http://www.w3.org/2000/svg" font-family="Arial, sans-serif" font-size="11">
  <defs><marker id="a8" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#666"/></marker></defs>
  <text x="350" y="20" text-anchor="middle" font-weight="bold" font-size="13" fill="#333">HunyuanVideo — dual-stream → single-stream DiT</text>
  <!-- inputs -->
  <rect x="30"  y="55" width="130" height="36" rx="6" fill="#e8f0fe" stroke="#4285f4" stroke-width="2"/>
  <text x="95" y="78" text-anchor="middle" fill="#1a73e8" font-weight="bold">video latent</text>
  <rect x="30" y="105" width="130" height="36" rx="6" fill="#fce8e6" stroke="#ea4335" stroke-width="2"/>
  <text x="95" y="128" text-anchor="middle" fill="#c5221f" font-weight="bold">MLLM text emb</text>
  <!-- Dual-stream stage -->
  <g transform="translate(180,40)">
    <rect x="0" y="0" width="230" height="170" rx="10" fill="#fff9f0" stroke="#f9ab00" stroke-width="2"/>
    <text x="115" y="20" text-anchor="middle" font-weight="bold" fill="#e37400">Dual-stream blocks</text>
    <text x="115" y="36" text-anchor="middle" fill="#888" font-size="10">separate Q/K/V + MLP per modality</text>
    <!-- video path -->
    <rect x="15" y="50" width="90" height="30" rx="5" fill="#e8f0fe" stroke="#4285f4"/>
    <text x="60" y="70" text-anchor="middle" fill="#1a73e8" font-size="11">video stream</text>
    <rect x="15" y="90" width="90" height="30" rx="5" fill="#e8f0fe" stroke="#4285f4"/>
    <text x="60" y="110" text-anchor="middle" fill="#1a73e8" font-size="10">× 19 layers</text>
    <!-- text path -->
    <rect x="125" y="50" width="90" height="30" rx="5" fill="#fce8e6" stroke="#ea4335"/>
    <text x="170" y="70" text-anchor="middle" fill="#c5221f" font-size="11">text stream</text>
    <rect x="125" y="90" width="90" height="30" rx="5" fill="#fce8e6" stroke="#ea4335"/>
    <text x="170" y="110" text-anchor="middle" fill="#c5221f" font-size="10">× 19 layers</text>
    <!-- joint attention arrows -->
    <line x1="105" y1="65" x2="125" y2="65" stroke="#34a853" stroke-width="1.5" marker-end="url(#a8)"/>
    <line x1="125" y1="105" x2="105" y2="105" stroke="#34a853" stroke-width="1.5" marker-end="url(#a8)"/>
    <text x="115" y="138" text-anchor="middle" fill="#1b5e20" font-size="10" font-weight="bold">joint attn only</text>
    <text x="115" y="153" text-anchor="middle" fill="#555" font-size="9">(modality-specific features)</text>
  </g>
  <line x1="415" y1="120" x2="455" y2="120" stroke="#666" stroke-width="1.5" marker-end="url(#a8)"/>
  <!-- Single-stream stage -->
  <g transform="translate(460,40)">
    <rect x="0" y="0" width="220" height="170" rx="10" fill="#fce4ec" stroke="#e91e63" stroke-width="2"/>
    <text x="110" y="20" text-anchor="middle" font-weight="bold" fill="#880e4f">Single-stream blocks</text>
    <text x="110" y="36" text-anchor="middle" fill="#888" font-size="10">concat → shared weights</text>
    <rect x="20" y="50" width="180" height="30" rx="5" fill="#fff" stroke="#e91e63"/>
    <text x="110" y="70" text-anchor="middle" fill="#880e4f" font-size="11">[ video ; text ] sequence</text>
    <rect x="20" y="90" width="180" height="30" rx="5" fill="#fff" stroke="#e91e63"/>
    <text x="110" y="110" text-anchor="middle" fill="#880e4f" font-size="11">shared Q/K/V + MLP</text>
    <text x="110" y="140" text-anchor="middle" fill="#880e4f" font-size="10" font-weight="bold">× 38 layers</text>
    <text x="110" y="155" text-anchor="middle" fill="#555" font-size="9">(deep cross-modal fusion)</text>
  </g>
  <!-- Translation table -->
  <g transform="translate(30,225)">
    <rect x="0" y="0" width="640" height="60" rx="8" fill="#f3f4f6" stroke="#bbb"/>
    <text x="320" y="18" text-anchor="middle" font-weight="bold" fill="#333">Translation table: FLUX (image) ↔ HunyuanVideo</text>
    <text x="10"  y="38" fill="#1a73e8"><tspan font-weight="bold">FLUX:</tspan> dual + single stream · <tspan fill="#c5221f">CLIP+OpenCLIP+T5</tspan> · Rectified Flow</text>
    <text x="10"  y="54" fill="#1a73e8"><tspan font-weight="bold">Hunyuan:</tspan> dual + single stream · <tspan fill="#c5221f">MLLM llava-llama-3-8b</tspan> · Flow Matching</text>
  </g>
</svg>
</center>

**Objective**: Flow Matching (v-prediction) — same shift as SD 1.5 → SD 3 for images.

[**Accent**] Translation table to Seminar 11:

| Image side (Seminar 11) | HunyuanVideo counterpart |
|---|---|
| FLUX dual+single stream | HunyuanVideo dual+single stream |
| Triple encoder (CLIP + OpenCLIP + T5) | **MLLM (llava-llama-3-8b)** |
| Rectified Flow v-prediction | Flow Matching v-prediction |

> **Reference**: Kong et al. "HunyuanVideo: A Systematic Framework for Large Video Generation Models", Tencent, Dec 2024.

### 2.5 Mochi-1 — Genmo (Oct 2024, OPEN, 10B)

- **AsymmDiT** (Asymmetric Diffusion Transformer): the visual stream has $\sim 4\times$ more parameters than the text stream.
- **Non-square QKV and output projections** unify the two streams at attention time.
- Memory-efficient at inference vs symmetric MMDiT for the same visual capacity.
- **Single T5-XXL** text encoder (no MLLM).
- **Full 3D attention** over **44,520 video tokens** — excellent quality, but prohibitive for long clips.
- **Output**: 480p, 5.4 s @ 30 fps.

<center>
<svg width="620" height="220" xmlns="http://www.w3.org/2000/svg" font-family="Arial, sans-serif" font-size="11">
  <defs><marker id="a9" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#666"/></marker></defs>
  <text x="310" y="18" text-anchor="middle" font-weight="bold" font-size="13" fill="#333">Mochi AsymmDiT — asymmetric streams, joint attention</text>
  <!-- Visual stream (big) -->
  <rect x="30" y="50" width="200" height="120" rx="8" fill="#e8f0fe" stroke="#4285f4" stroke-width="2"/>
  <text x="130" y="70" text-anchor="middle" fill="#1a73e8" font-weight="bold" font-size="12">Visual stream</text>
  <text x="130" y="90" text-anchor="middle" fill="#555" font-size="10">~80% of parameters</text>
  <rect x="50" y="105" width="160" height="20" rx="3" fill="#cfe2ff" stroke="#4285f4"/>
  <rect x="50" y="130" width="160" height="20" rx="3" fill="#cfe2ff" stroke="#4285f4"/>
  <text x="130" y="162" text-anchor="middle" fill="#1a73e8" font-size="10">44,520 video tokens · full 3D attn</text>
  <!-- Text stream (small) -->
  <rect x="430" y="80" width="160" height="60" rx="8" fill="#fce8e6" stroke="#ea4335" stroke-width="2"/>
  <text x="510" y="100" text-anchor="middle" fill="#c5221f" font-weight="bold" font-size="12">Text stream</text>
  <text x="510" y="118" text-anchor="middle" fill="#555" font-size="10">~20% of parameters</text>
  <text x="510" y="150" text-anchor="middle" fill="#c5221f" font-size="10">T5-XXL 4096-dim</text>
  <!-- Joint -->
  <rect x="245" y="95" width="170" height="36" rx="6" fill="#fef7e0" stroke="#f9ab00" stroke-width="2"/>
  <text x="330" y="118" text-anchor="middle" fill="#e37400" font-weight="bold" font-size="11">non-square Q/K/V proj</text>
  <line x1="230" y1="113" x2="245" y2="113" stroke="#4285f4" stroke-width="1.5" marker-end="url(#a9)"/>
  <line x1="430" y1="113" x2="415" y2="113" stroke="#ea4335" stroke-width="1.5" marker-end="url(#a9)"/>
  <line x1="330" y1="131" x2="330" y2="160" stroke="#666" stroke-width="1.5" marker-end="url(#a9)"/>
  <rect x="235" y="160" width="190" height="34" rx="6" fill="#e8f5e9" stroke="#34a853" stroke-width="2"/>
  <text x="330" y="181" text-anchor="middle" fill="#1b5e20" font-weight="bold" font-size="11">Unified Joint Self-Attn</text>
</svg>
</center>

Mochi's design statement: "most of the work happens on the visual side, so spend the parameters there".

### 2.6 LTX-Video — Lightricks (Dec 2024, OPEN, arXiv:2501.00103)

The **fastest open video model**. Two related ideas:

- **Extreme 1:192 compression VAE**: a single latent token represents a $32 \times 32 \times 8$ pixel tubelet. Two orders of magnitude more aggressive than the usual $8\times 8\times 4$.
- **Patchification moved from transformer into the VAE itself**. The VAE outputs tokens directly — no tokenization layer in the DiT.
- **Decoder performs the final denoising step** — the line between VAE and diffusion is blurred.

<center>
<svg width="680" height="210" xmlns="http://www.w3.org/2000/svg" font-family="Arial, sans-serif" font-size="11">
  <text x="340" y="18" text-anchor="middle" font-weight="bold" font-size="13" fill="#333">Compression ratios — LTX vs the industry standard</text>
  <!-- Standard -->
  <g transform="translate(30,45)">
    <text x="140" y="-5" text-anchor="middle" fill="#333" font-size="12" font-weight="bold">Standard VAE (CogVideoX, Hunyuan)</text>
    <rect x="0" y="5" width="100" height="100" rx="3" fill="#e8f0fe" stroke="#4285f4" stroke-width="1.5"/>
    <text x="50" y="60" text-anchor="middle" fill="#1a73e8" font-size="10">8×8×4 pixels</text>
    <text x="155" y="60" text-anchor="middle" fill="#666" font-size="20">→</text>
    <rect x="190" y="45" width="20" height="20" rx="2" fill="#ede7f6" stroke="#7c4dff"/>
    <text x="200" y="82" text-anchor="middle" fill="#7627bb" font-size="10">1 token</text>
    <text x="140" y="130" text-anchor="middle" fill="#888" font-size="10">1 : 256 ratio</text>
  </g>
  <!-- Divider -->
  <line x1="340" y1="45" x2="340" y2="165" stroke="#ddd" stroke-width="1.5" stroke-dasharray="5,3"/>
  <!-- LTX -->
  <g transform="translate(370,45)">
    <text x="140" y="-5" text-anchor="middle" fill="#333" font-size="12" font-weight="bold">LTX extreme VAE</text>
    <rect x="0"  y="5"  width="110" height="110" rx="3" fill="#fce8e6" stroke="#ea4335" stroke-width="1.5"/>
    <rect x="4"  y="9"  width="110" height="110" rx="3" fill="#fce8e6" stroke="#ea4335" stroke-width="1.5"/>
    <rect x="8"  y="13" width="110" height="110" rx="3" fill="#fce8e6" stroke="#ea4335" stroke-width="1.5"/>
    <rect x="12" y="17" width="110" height="110" rx="3" fill="#fce8e6" stroke="#ea4335" stroke-width="1.5"/>
    <text x="70" y="75" text-anchor="middle" fill="#c5221f" font-size="10">32×32×8 pixels</text>
    <text x="150" y="75" text-anchor="middle" fill="#666" font-size="20">→</text>
    <rect x="185" y="60" width="22" height="22" rx="2" fill="#ede7f6" stroke="#7c4dff"/>
    <text x="196" y="100" text-anchor="middle" fill="#7627bb" font-size="10">1 token</text>
    <text x="140" y="145" text-anchor="middle" fill="#c5221f" font-weight="bold" font-size="11">1 : 192 × larger tubelets</text>
  </g>
  <!-- Speed badge -->
  <rect x="30" y="178" width="620" height="24" rx="6" fill="#fff9f0" stroke="#f9ab00"/>
  <text x="340" y="194" text-anchor="middle" fill="#e37400" font-weight="bold">5 s @ 768×512 @ 24 fps → 2 s on H100 · 121 frames → 11 s on RTX 4090</text>
</svg>
</center>

**Speed**:
- 5 s @ 24 fps @ 768×512 in **2 s on an H100**.
- 121 frames in 11 s on an RTX 4090.

**VRAM**: $\sim$10 GB nominal, down to $\sim$6 GB with Q8 quantisation — **T4-possible**.

> **Reference**: HaCohen et al. "LTX-Video: Realtime Video Latent Diffusion", arXiv:2501.00103.

### 2.7 Wan 2.1 / Wan 2.2 — Alibaba (2025, OPEN, Apache 2.0)

Alibaba's open release with a consumer-grade variant and a next-gen MoE variant.

**Wan 2.1**:
- `Wan2.1-T2V-1.3B` — $\sim 8.2$ GB VRAM in bf16 → the **only model in this seminar that actually fits free Colab T4**. This is our main live demo.
- `Wan2.1-T2V-14B` — theory only for us.
- Standard DiT + cross-attention, 3D VAE, T5 text encoder, Flow Matching objective.

**Wan 2.2 — MoE along the denoising trajectory** (the new idea):

<center>
<svg width="680" height="250" xmlns="http://www.w3.org/2000/svg" font-family="Arial, sans-serif" font-size="11">
  <defs><marker id="aWa" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#666"/></marker></defs>
  <text x="340" y="18" text-anchor="middle" font-weight="bold" font-size="13" fill="#333">Wan 2.2 — MoE along the denoising trajectory</text>
  <!-- Horizontal axis: noise to data -->
  <line x1="60"  y1="130" x2="620" y2="130" stroke="#666" stroke-width="1.5"/>
  <text x="340" y="155" text-anchor="middle" fill="#555" font-size="11">denoising timestep t</text>
  <text x="50"  y="134" text-anchor="end" fill="#888" font-size="10">noise</text>
  <text x="630" y="134" fill="#888" font-size="10">data</text>
  <!-- Switch point -->
  <line x1="340" y1="80" x2="340" y2="175" stroke="#9334e6" stroke-width="1.5" stroke-dasharray="4,3"/>
  <text x="340" y="72" text-anchor="middle" fill="#7627bb" font-size="10" font-weight="bold">switch (SNR drops ×½)</text>
  <!-- High noise expert -->
  <rect x="60"  y="90" width="260" height="36" rx="6" fill="#fce8e6" stroke="#ea4335" stroke-width="2"/>
  <text x="190" y="110" text-anchor="middle" fill="#c5221f" font-weight="bold">HIGH-NOISE expert — 14B</text>
  <text x="190" y="124" text-anchor="middle" fill="#555" font-size="10">composition, layout</text>
  <!-- Low noise expert -->
  <rect x="360" y="90" width="260" height="36" rx="6" fill="#e8f5e9" stroke="#34a853" stroke-width="2"/>
  <text x="490" y="110" text-anchor="middle" fill="#1b5e20" font-weight="bold">LOW-NOISE expert — 14B</text>
  <text x="490" y="124" text-anchor="middle" fill="#555" font-size="10">detail, texture, color</text>
  <!-- Tick marks -->
  <circle cx="60"  cy="130" r="4" fill="#ea4335"/>
  <circle cx="340" cy="130" r="4" fill="#9334e6"/>
  <circle cx="620" cy="130" r="4" fill="#34a853"/>
  <!-- Summary -->
  <rect x="110" y="185" width="460" height="56" rx="8" fill="#fff9f0" stroke="#f9ab00" stroke-width="1.5"/>
  <text x="340" y="204" text-anchor="middle" fill="#e37400" font-weight="bold" font-size="12">total 27B params · only 14B active per step</text>
  <text x="340" y="221" text-anchor="middle" fill="#555" font-size="11">memory/compute parity with a dense 14B model</text>
  <text x="340" y="236" text-anchor="middle" fill="#555" font-size="10" font-style="italic">new axis: MoE along noise — not along layers or tokens</text>
</svg>
</center>

- Two $\sim$14B experts, **switched mid-rollout** by signal-to-noise ratio (when the noise proportion drops by half).
- Total 27B params; only 14B are **active** per step — memory/compute parity with a 14B dense model.
- Training objective: Flow Matching.

[**Accent**] Observation from Seminar 11 carried over: image MoE inside a layer (Mixture-of-Experts transformer) never really caught on; **MoE along the noise axis** is a new axis that only makes sense for diffusion/FM.

### 2.8 Veo 3 — Google DeepMind (2025, closed API)

- **Latent Diffusion Transformer** jointly generating **video + audio**.
- Diffusion applied to a **unified token sequence** interleaving spatiotemporal video latents with temporal audio latents — a single attention stack synchronises lips, footsteps, and ambience to pixels.
- Sub-components (community estimates): 12B transformer producing keyframes every 2 s, a 28B U-Net interpolator between keyframes, and a 9B audio engine.
- Reported lip-sync accuracy **< 120 ms**.

<center>
<svg width="680" height="170" xmlns="http://www.w3.org/2000/svg" font-family="Arial, sans-serif" font-size="11">
  <text x="340" y="18" text-anchor="middle" font-weight="bold" font-size="13" fill="#333">Veo 3 — interleaved A/V token sequence, single attention stack</text>
  <!-- Tokens -->
  <g transform="translate(30,40)">
    <!-- Repeat v a v v a v a v v ... -->
    <rect x="0"   y="0" width="40" height="30" rx="3" fill="#e8f0fe" stroke="#4285f4"/>
    <text x="20"  y="20" text-anchor="middle" fill="#1a73e8" font-weight="bold" font-size="11">V</text>
    <rect x="45"  y="0" width="40" height="30" rx="3" fill="#e8f0fe" stroke="#4285f4"/>
    <text x="65"  y="20" text-anchor="middle" fill="#1a73e8" font-weight="bold" font-size="11">V</text>
    <rect x="90"  y="0" width="40" height="30" rx="3" fill="#fde7f3" stroke="#e91e63"/>
    <text x="110" y="20" text-anchor="middle" fill="#880e4f" font-weight="bold" font-size="11">A</text>
    <rect x="135" y="0" width="40" height="30" rx="3" fill="#e8f0fe" stroke="#4285f4"/>
    <text x="155" y="20" text-anchor="middle" fill="#1a73e8" font-weight="bold" font-size="11">V</text>
    <rect x="180" y="0" width="40" height="30" rx="3" fill="#e8f0fe" stroke="#4285f4"/>
    <text x="200" y="20" text-anchor="middle" fill="#1a73e8" font-weight="bold" font-size="11">V</text>
    <rect x="225" y="0" width="40" height="30" rx="3" fill="#fde7f3" stroke="#e91e63"/>
    <text x="245" y="20" text-anchor="middle" fill="#880e4f" font-weight="bold" font-size="11">A</text>
    <rect x="270" y="0" width="40" height="30" rx="3" fill="#e8f0fe" stroke="#4285f4"/>
    <text x="290" y="20" text-anchor="middle" fill="#1a73e8" font-weight="bold" font-size="11">V</text>
    <rect x="315" y="0" width="40" height="30" rx="3" fill="#fde7f3" stroke="#e91e63"/>
    <text x="335" y="20" text-anchor="middle" fill="#880e4f" font-weight="bold" font-size="11">A</text>
    <rect x="360" y="0" width="40" height="30" rx="3" fill="#e8f0fe" stroke="#4285f4"/>
    <text x="380" y="20" text-anchor="middle" fill="#1a73e8" font-weight="bold" font-size="11">V</text>
    <rect x="405" y="0" width="40" height="30" rx="3" fill="#e8f0fe" stroke="#4285f4"/>
    <text x="425" y="20" text-anchor="middle" fill="#1a73e8" font-weight="bold" font-size="11">V</text>
    <rect x="450" y="0" width="40" height="30" rx="3" fill="#fde7f3" stroke="#e91e63"/>
    <text x="470" y="20" text-anchor="middle" fill="#880e4f" font-weight="bold" font-size="11">A</text>
    <rect x="495" y="0" width="40" height="30" rx="3" fill="#e8f0fe" stroke="#4285f4"/>
    <text x="515" y="20" text-anchor="middle" fill="#1a73e8" font-weight="bold" font-size="11">V</text>
    <rect x="540" y="0" width="40" height="30" rx="3" fill="#fde7f3" stroke="#e91e63"/>
    <text x="560" y="20" text-anchor="middle" fill="#880e4f" font-weight="bold" font-size="11">A</text>
    <rect x="585" y="0" width="40" height="30" rx="3" fill="#e8f0fe" stroke="#4285f4"/>
    <text x="605" y="20" text-anchor="middle" fill="#1a73e8" font-weight="bold" font-size="11">V</text>
  </g>
  <text x="340" y="92" text-anchor="middle" fill="#333" font-size="11">unified token sequence (V = video latent, A = audio latent)</text>
  <!-- Attention stack box -->
  <rect x="130" y="105" width="420" height="40" rx="8" fill="#fef7e0" stroke="#f9ab00" stroke-width="2"/>
  <text x="340" y="124" text-anchor="middle" fill="#e37400" font-weight="bold">Single Latent Diffusion Transformer</text>
  <text x="340" y="140" text-anchor="middle" fill="#555" font-size="10">sub-100 ms lip-sync · 12B keyframe + 28B interp + 9B audio</text>
</svg>
</center>

Veo 3 is the first model to make "joint A/V generation" a first-class design goal, not an afterthought.

> **Reference**: DeepMind "Veo 3 technical report", 2025.

### 2.9 The Summary Table

| Model | Year | Params | Backbone | VAE | Text encoder | Objective | Key innovation |
|---|---|---|---|---|---|---|---|
| VDM | 2022 | $<1$B | 3D U-Net | pixel | BERT-style | DDPM $\epsilon$ | Factorized space-time attn |
| SVD | 2023 | $\sim1.5$B | 2D + temporal U-Net | SD VAE | CLIP | DDPM | Temporal layers in LDM |
| Lumiere | 2024 | $\sim3$B | Space-Time U-Net | pixel | T5 | DDPM | Single-pass full-video |
| **Sora** | 2024 | $\sim3{-}10$B | DiT | in-house 3D | T5-like | DDPM-era | Spacetime patches, variable shape |
| **Kling** | 2024 | — | DiT + 3D VAE | in-house | — | — | 2-min 1080p (full 3D attn) |
| **CogVideoX** | 2024 | 2B / 5B | **Expert MMDiT** | 3D causal $8\times8\times4$ | T5-XXL | v-pred | Per-modality AdaLN, 3D RoPE |
| **Mochi-1** | 2024 | 10B | **AsymmDiT** | 3D VAE | T5-XXL | Flow Matching | Asymmetric 4× visual stream |
| **HunyuanVideo** | 2024 | 13B | **Dual→Single DiT** | 3D causal | **MLLM (llava-llama-3-8b)** | Flow Matching | FLUX blocks + MLLM text |
| **LTX-Video** | 2024 | 2B / 13B | DiT | **1:192 VAE** | T5-XXL | Flow Matching | Patchify inside VAE |
| **Wan 2.1** | 2025 | 1.3B / 14B | DiT + cross-attn | Wan-VAE | T5 | Flow Matching | Consumer-grade 1.3B |
| **Wan 2.2** | 2025 | 27B (14B active) | **DiT MoE along noise** | Wan-VAE | T5 | Flow Matching | High/low-noise experts |
| **Veo 3** | 2025 | 12B+28B+9B | LDT | spatiotemporal + audio | — | — | **Joint audio-video** |

<center>
<svg width="720" height="290" xmlns="http://www.w3.org/2000/svg" font-family="Arial, sans-serif" font-size="11">
  <text x="360" y="20" text-anchor="middle" font-weight="bold" font-size="14" fill="#333">Three overarching narratives — video recapitulates image</text>
  <!-- Axis 1: Architecture -->
  <g transform="translate(30,50)">
    <text x="0" y="0" font-weight="bold" fill="#1a73e8" font-size="12">① Architecture</text>
    <rect x="0"   y="10" width="90"  height="28" rx="4" fill="#e8f0fe" stroke="#4285f4"/>
    <text x="45"  y="28" text-anchor="middle" fill="#1a73e8" font-size="10">factorized UNet</text>
    <text x="100" y="28" fill="#888" font-size="12">→</text>
    <rect x="115" y="10" width="70"  height="28" rx="4" fill="#e8f0fe" stroke="#4285f4"/>
    <text x="150" y="28" text-anchor="middle" fill="#1a73e8" font-size="10">full UNet</text>
    <text x="195" y="28" fill="#888" font-size="12">→</text>
    <rect x="210" y="10" width="60"  height="28" rx="4" fill="#fef7e0" stroke="#f9ab00"/>
    <text x="240" y="28" text-anchor="middle" fill="#e37400" font-size="10">DiT</text>
    <text x="280" y="28" fill="#888" font-size="12">→</text>
    <rect x="295" y="10" width="80"  height="28" rx="4" fill="#fef7e0" stroke="#f9ab00"/>
    <text x="335" y="28" text-anchor="middle" fill="#e37400" font-size="10">MMDiT</text>
    <text x="385" y="28" fill="#888" font-size="12">→</text>
    <rect x="400" y="10" width="110" height="28" rx="4" fill="#fce4ec" stroke="#e91e63"/>
    <text x="455" y="28" text-anchor="middle" fill="#880e4f" font-size="10">dual+single</text>
    <text x="520" y="28" fill="#888" font-size="12">→</text>
    <rect x="535" y="10" width="120" height="28" rx="4" fill="#fce4ec" stroke="#e91e63"/>
    <text x="595" y="28" text-anchor="middle" fill="#880e4f" font-size="10">MoE along noise</text>
  </g>
  <!-- Axis 2: VAE -->
  <g transform="translate(30,110)">
    <text x="0" y="0" font-weight="bold" fill="#e37400" font-size="12">② VAE compression</text>
    <rect x="0"   y="10" width="70"  height="28" rx="4" fill="#fff3e0" stroke="#ff9800"/>
    <text x="35"  y="28" text-anchor="middle" fill="#e65100" font-size="10">pixel</text>
    <text x="80"  y="28" fill="#888" font-size="12">→</text>
    <rect x="95"  y="10" width="90"  height="28" rx="4" fill="#fff3e0" stroke="#ff9800"/>
    <text x="140" y="28" text-anchor="middle" fill="#e65100" font-size="10">2D image VAE</text>
    <text x="195" y="28" fill="#888" font-size="12">→</text>
    <rect x="210" y="10" width="140" height="28" rx="4" fill="#fff3e0" stroke="#ff9800"/>
    <text x="280" y="28" text-anchor="middle" fill="#e65100" font-size="10">3D causal 8×8×4</text>
    <text x="360" y="28" fill="#888" font-size="12">→</text>
    <rect x="375" y="10" width="150" height="28" rx="4" fill="#fce8e6" stroke="#ea4335"/>
    <text x="450" y="28" text-anchor="middle" fill="#c5221f" font-size="10">extreme 1:192 (LTX)</text>
  </g>
  <!-- Axis 3: Objective -->
  <g transform="translate(30,170)">
    <text x="0" y="0" font-weight="bold" fill="#34a853" font-size="12">③ Training objective</text>
    <rect x="0"   y="10" width="140" height="28" rx="4" fill="#e8f0fe" stroke="#4285f4"/>
    <text x="70"  y="28" text-anchor="middle" fill="#1a73e8" font-size="10">DDPM ε-prediction</text>
    <text x="150" y="28" fill="#888" font-size="12">→</text>
    <rect x="165" y="10" width="180" height="28" rx="4" fill="#e8f5e9" stroke="#34a853"/>
    <text x="255" y="28" text-anchor="middle" fill="#1b5e20" font-size="10">Rectified / Flow Matching v-pred</text>
  </g>
  <!-- Caption -->
  <rect x="30" y="230" width="660" height="44" rx="8" fill="#f3f4f6" stroke="#bbb"/>
  <text x="360" y="249" text-anchor="middle" fill="#333" font-weight="bold" font-size="12">Exact parallel to Seminar 11 image evolution</text>
  <text x="360" y="266" text-anchor="middle" fill="#555" font-size="11">Video is ~6 months behind images on every axis.</text>
</svg>
</center>

[**Important — three overarching narratives**]:

1. **Architecture**: Factorized U-Net $\to$ full-3D U-Net $\to$ DiT (Sora) $\to$ MMDiT (CogVideoX) $\to$ Dual+Single-stream (HunyuanVideo) $\to$ MoE-along-noise (Wan 2.2). **Exact parallel to the image evolution `UNet → DiT → MMDiT → FLUX` from Seminar 11.**

2. **VAE compression**: pixel $\to$ 2D image VAE (SVD) $\to$ 3D causal VAE $8\times 8\times 4$ (everyone modern) $\to$ extreme 1:192 (LTX).

3. **Training objective**: DDPM $\epsilon$-prediction $\to$ Flow Matching v-prediction. Video followed image with a ~6-month lag.

## 3. Autoregressive Video — Next-Frame Prediction and World Models

So far every model we have seen is a **parallel diffusion** model: the whole clip is generated from noise at once. There is a second paradigm — **autoregressive video** — where frames (or blocks of frames) are generated one at a time, conditioning on the past. This is the paradigm of LLMs lifted to video.

### 3.1 Two Paradigms Side by Side

| Aspect | **Parallel diffusion** | **Autoregressive** |
|---|---|---|
| Generation | All frames together | One frame (or block) at a time |
| Attention mask in time | Bidirectional | Causal |
| Clip length | Fixed (training budget) | Arbitrary, streaming |
| Error accumulation | None within a clip | Compounds over rollout |
| Temporal consistency | Strong on short clips | Weak on long horizons |
| Natural use-cases | Short high-quality clips | World models, games, streaming |
| Examples | Sora, CogVideoX, HunyuanVideo, LTX | VideoGPT, MAGVIT-v2, Genie 2, GameNGen |

<center>
<svg width="700" height="220" xmlns="http://www.w3.org/2000/svg" font-family="Arial, sans-serif" font-size="11">
  <defs><marker id="aP" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#666"/></marker></defs>
  <text x="350" y="18" text-anchor="middle" font-weight="bold" font-size="13" fill="#333">Parallel diffusion vs Autoregressive</text>
  <!-- Parallel -->
  <g transform="translate(20,40)">
    <text x="155" y="0" text-anchor="middle" fill="#1a73e8" font-weight="bold" font-size="12">Parallel diffusion</text>
    <rect x="0"   y="15" width="50" height="50" rx="4" fill="#e8f0fe" stroke="#4285f4"/>
    <rect x="55"  y="15" width="50" height="50" rx="4" fill="#e8f0fe" stroke="#4285f4"/>
    <rect x="110" y="15" width="50" height="50" rx="4" fill="#e8f0fe" stroke="#4285f4"/>
    <rect x="165" y="15" width="50" height="50" rx="4" fill="#e8f0fe" stroke="#4285f4"/>
    <rect x="220" y="15" width="50" height="50" rx="4" fill="#e8f0fe" stroke="#4285f4"/>
    <text x="135" y="42" text-anchor="middle" fill="#1a73e8" font-size="10">all frames at once</text>
    <!-- bidirectional arrows -->
    <line x1="25"  y1="85" x2="245" y2="85" stroke="#4285f4" stroke-width="1.3" marker-end="url(#aP)" marker-start="url(#aP)"/>
    <text x="135" y="102" text-anchor="middle" fill="#555" font-size="10">bidirectional attention in time</text>
    <rect x="0" y="115" width="270" height="38" rx="5" fill="#e8f5e9" stroke="#34a853" stroke-width="1.3"/>
    <text x="135" y="132" text-anchor="middle" fill="#1b5e20" font-weight="bold" font-size="11">+ strong coherence</text>
    <text x="135" y="146" text-anchor="middle" fill="#555" font-size="10">− fixed length (training budget)</text>
  </g>
  <!-- Divider -->
  <line x1="350" y1="35" x2="350" y2="200" stroke="#ddd" stroke-width="1.5" stroke-dasharray="5,3"/>
  <!-- Autoregressive -->
  <g transform="translate(380,40)">
    <text x="155" y="0" text-anchor="middle" fill="#c5221f" font-weight="bold" font-size="12">Autoregressive</text>
    <rect x="0"   y="15" width="50" height="50" rx="4" fill="#fce8e6" stroke="#ea4335"/>
    <rect x="55"  y="15" width="50" height="50" rx="4" fill="#fce8e6" stroke="#ea4335" fill-opacity="0.75"/>
    <rect x="110" y="15" width="50" height="50" rx="4" fill="#fce8e6" stroke="#ea4335" fill-opacity="0.55"/>
    <rect x="165" y="15" width="50" height="50" rx="4" fill="#fce8e6" stroke="#ea4335" fill-opacity="0.35"/>
    <rect x="220" y="15" width="50" height="50" rx="4" fill="#fff" stroke="#ea4335" stroke-dasharray="3,2"/>
    <text x="135" y="42" text-anchor="middle" fill="#c5221f" font-size="10">one at a time</text>
    <!-- causal arrows -->
    <line x1="25"  y1="85" x2="80"  y2="85" stroke="#ea4335" stroke-width="1.3" marker-end="url(#aP)"/>
    <line x1="80"  y1="85" x2="135" y2="85" stroke="#ea4335" stroke-width="1.3" marker-end="url(#aP)"/>
    <line x1="135" y1="85" x2="190" y2="85" stroke="#ea4335" stroke-width="1.3" marker-end="url(#aP)"/>
    <line x1="190" y1="85" x2="245" y2="85" stroke="#ea4335" stroke-width="1.3" marker-end="url(#aP)"/>
    <text x="135" y="102" text-anchor="middle" fill="#555" font-size="10">causal attention, past only</text>
    <rect x="0" y="115" width="270" height="38" rx="5" fill="#fde7f3" stroke="#e91e63" stroke-width="1.3"/>
    <text x="135" y="132" text-anchor="middle" fill="#880e4f" font-weight="bold" font-size="11">+ streaming, unbounded length</text>
    <text x="135" y="146" text-anchor="middle" fill="#555" font-size="10">− error accumulates on long rollouts</text>
  </g>
</svg>
</center>

The two regimes are converging (see Genie 2 below), but the paradigmatic difference is worth understanding on its own.

### 3.2 Causal Attention in Time

The only architectural change needed to go from parallel → autoregressive is an **attention mask**. Within a single frame the model can attend freely in space; across frames, the mask is **causal**:

$$\text{Attn}(Q, K, V) = \text{softmax}\!\left(\frac{QK^{\top}}{\sqrt{d}} + M\right)V$$

$$M_{ij} = \begin{cases} 0 & \text{if }\text{frame}(i) \geq \text{frame}(j) \\ -\infty & \text{otherwise} \end{cases}$$

<center>
<svg width="440" height="360" xmlns="http://www.w3.org/2000/svg" font-family="Arial, sans-serif" font-size="11">
  <text x="220" y="20" text-anchor="middle" font-weight="bold" font-size="13" fill="#333">Attention mask over 4 frames (3 tokens per frame)</text>
  <!-- Grid 12x12 -->
  <g transform="translate(60,45)">
    <!-- headers -->
    <text x="145"  y="-10" text-anchor="middle" fill="#333" font-size="11">key tokens (j) →</text>
    <text x="-20" y="150" text-anchor="middle" fill="#333" font-size="11" transform="rotate(-90 -20,150)">query tokens (i) →</text>
  </g>
  <g transform="translate(70,48)">
    <!-- 12x12 grid: 4 frames of 3 tokens. allow attention when frame(i) >= frame(j) -->
    <!-- build cells -->
    <!-- frame assignments: token k in [0,11], frame = k // 3 -->
    <!-- row i, col j: if i//3 >= j//3 → green else red -->
    <g>
      <!-- row 0 (frame 0) -->
      <rect x="0"   y="0" width="25" height="25" fill="#c8e6c9" stroke="#34a853" stroke-width="0.5"/>
      <rect x="25"  y="0" width="25" height="25" fill="#c8e6c9" stroke="#34a853" stroke-width="0.5"/>
      <rect x="50"  y="0" width="25" height="25" fill="#c8e6c9" stroke="#34a853" stroke-width="0.5"/>
      <rect x="75"  y="0" width="25" height="25" fill="#fbd0d0" stroke="#ea4335" stroke-width="0.5"/>
      <rect x="100" y="0" width="25" height="25" fill="#fbd0d0" stroke="#ea4335" stroke-width="0.5"/>
      <rect x="125" y="0" width="25" height="25" fill="#fbd0d0" stroke="#ea4335" stroke-width="0.5"/>
      <rect x="150" y="0" width="25" height="25" fill="#fbd0d0" stroke="#ea4335" stroke-width="0.5"/>
      <rect x="175" y="0" width="25" height="25" fill="#fbd0d0" stroke="#ea4335" stroke-width="0.5"/>
      <rect x="200" y="0" width="25" height="25" fill="#fbd0d0" stroke="#ea4335" stroke-width="0.5"/>
      <rect x="225" y="0" width="25" height="25" fill="#fbd0d0" stroke="#ea4335" stroke-width="0.5"/>
      <rect x="250" y="0" width="25" height="25" fill="#fbd0d0" stroke="#ea4335" stroke-width="0.5"/>
      <rect x="275" y="0" width="25" height="25" fill="#fbd0d0" stroke="#ea4335" stroke-width="0.5"/>
      <!-- row 1 -->
      <rect x="0"   y="25" width="25" height="25" fill="#c8e6c9" stroke="#34a853" stroke-width="0.5"/>
      <rect x="25"  y="25" width="25" height="25" fill="#c8e6c9" stroke="#34a853" stroke-width="0.5"/>
      <rect x="50"  y="25" width="25" height="25" fill="#c8e6c9" stroke="#34a853" stroke-width="0.5"/>
      <rect x="75"  y="25" width="25" height="25" fill="#fbd0d0" stroke="#ea4335" stroke-width="0.5"/>
      <rect x="100" y="25" width="25" height="25" fill="#fbd0d0" stroke="#ea4335" stroke-width="0.5"/>
      <rect x="125" y="25" width="25" height="25" fill="#fbd0d0" stroke="#ea4335" stroke-width="0.5"/>
      <rect x="150" y="25" width="25" height="25" fill="#fbd0d0" stroke="#ea4335" stroke-width="0.5"/>
      <rect x="175" y="25" width="25" height="25" fill="#fbd0d0" stroke="#ea4335" stroke-width="0.5"/>
      <rect x="200" y="25" width="25" height="25" fill="#fbd0d0" stroke="#ea4335" stroke-width="0.5"/>
      <rect x="225" y="25" width="25" height="25" fill="#fbd0d0" stroke="#ea4335" stroke-width="0.5"/>
      <rect x="250" y="25" width="25" height="25" fill="#fbd0d0" stroke="#ea4335" stroke-width="0.5"/>
      <rect x="275" y="25" width="25" height="25" fill="#fbd0d0" stroke="#ea4335" stroke-width="0.5"/>
      <!-- row 2 -->
      <rect x="0"   y="50" width="25" height="25" fill="#c8e6c9" stroke="#34a853" stroke-width="0.5"/>
      <rect x="25"  y="50" width="25" height="25" fill="#c8e6c9" stroke="#34a853" stroke-width="0.5"/>
      <rect x="50"  y="50" width="25" height="25" fill="#c8e6c9" stroke="#34a853" stroke-width="0.5"/>
      <rect x="75"  y="50" width="25" height="25" fill="#fbd0d0" stroke="#ea4335" stroke-width="0.5"/>
      <rect x="100" y="50" width="25" height="25" fill="#fbd0d0" stroke="#ea4335" stroke-width="0.5"/>
      <rect x="125" y="50" width="25" height="25" fill="#fbd0d0" stroke="#ea4335" stroke-width="0.5"/>
      <rect x="150" y="50" width="25" height="25" fill="#fbd0d0" stroke="#ea4335" stroke-width="0.5"/>
      <rect x="175" y="50" width="25" height="25" fill="#fbd0d0" stroke="#ea4335" stroke-width="0.5"/>
      <rect x="200" y="50" width="25" height="25" fill="#fbd0d0" stroke="#ea4335" stroke-width="0.5"/>
      <rect x="225" y="50" width="25" height="25" fill="#fbd0d0" stroke="#ea4335" stroke-width="0.5"/>
      <rect x="250" y="50" width="25" height="25" fill="#fbd0d0" stroke="#ea4335" stroke-width="0.5"/>
      <rect x="275" y="50" width="25" height="25" fill="#fbd0d0" stroke="#ea4335" stroke-width="0.5"/>
      <!-- rows 3-5 (frame 1) -->
      <rect x="0"   y="75" width="150" height="25" fill="#c8e6c9" stroke="#34a853" stroke-width="0.5"/>
      <rect x="150" y="75" width="150" height="25" fill="#fbd0d0" stroke="#ea4335" stroke-width="0.5"/>
      <rect x="0"   y="100" width="150" height="25" fill="#c8e6c9" stroke="#34a853" stroke-width="0.5"/>
      <rect x="150" y="100" width="150" height="25" fill="#fbd0d0" stroke="#ea4335" stroke-width="0.5"/>
      <rect x="0"   y="125" width="150" height="25" fill="#c8e6c9" stroke="#34a853" stroke-width="0.5"/>
      <rect x="150" y="125" width="150" height="25" fill="#fbd0d0" stroke="#ea4335" stroke-width="0.5"/>
      <!-- rows 6-8 (frame 2) -->
      <rect x="0"   y="150" width="225" height="25" fill="#c8e6c9" stroke="#34a853" stroke-width="0.5"/>
      <rect x="225" y="150" width="75"  height="25" fill="#fbd0d0" stroke="#ea4335" stroke-width="0.5"/>
      <rect x="0"   y="175" width="225" height="25" fill="#c8e6c9" stroke="#34a853" stroke-width="0.5"/>
      <rect x="225" y="175" width="75"  height="25" fill="#fbd0d0" stroke="#ea4335" stroke-width="0.5"/>
      <rect x="0"   y="200" width="225" height="25" fill="#c8e6c9" stroke="#34a853" stroke-width="0.5"/>
      <rect x="225" y="200" width="75"  height="25" fill="#fbd0d0" stroke="#ea4335" stroke-width="0.5"/>
      <!-- rows 9-11 (frame 3) -->
      <rect x="0" y="225" width="300" height="25" fill="#c8e6c9" stroke="#34a853" stroke-width="0.5"/>
      <rect x="0" y="250" width="300" height="25" fill="#c8e6c9" stroke="#34a853" stroke-width="0.5"/>
      <rect x="0" y="275" width="300" height="25" fill="#c8e6c9" stroke="#34a853" stroke-width="0.5"/>
    </g>
    <!-- Frame boundaries -->
    <line x1="75"  y1="0" x2="75"  y2="300" stroke="#333" stroke-width="1"/>
    <line x1="150" y1="0" x2="150" y2="300" stroke="#333" stroke-width="1"/>
    <line x1="225" y1="0" x2="225" y2="300" stroke="#333" stroke-width="1"/>
    <line x1="0" y1="75"  x2="300" y2="75"  stroke="#333" stroke-width="1"/>
    <line x1="0" y1="150" x2="300" y2="150" stroke="#333" stroke-width="1"/>
    <line x1="0" y1="225" x2="300" y2="225" stroke="#333" stroke-width="1"/>
    <!-- Frame labels -->
    <text x="37"  y="-5" text-anchor="middle" fill="#555" font-size="10">frame 0</text>
    <text x="112" y="-5" text-anchor="middle" fill="#555" font-size="10">frame 1</text>
    <text x="187" y="-5" text-anchor="middle" fill="#555" font-size="10">frame 2</text>
    <text x="262" y="-5" text-anchor="middle" fill="#555" font-size="10">frame 3</text>
  </g>
  <!-- Legend -->
  <rect x="310" y="60" width="14" height="14" fill="#c8e6c9" stroke="#34a853"/>
  <text x="330" y="72" fill="#333" font-size="10">allowed (past + same frame)</text>
  <rect x="310" y="80" width="14" height="14" fill="#fbd0d0" stroke="#ea4335"/>
  <text x="330" y="92" fill="#333" font-size="10">masked (future frame)</text>
</svg>
</center>

This is exactly the decoder-LLM causal mask, lifted to a 3D spacetime token grid. A token at frame $t$ can see all tokens from frames $\leq t$ (including its own spatial neighbours) but never any token from frame $> t$.

### 3.3 VideoGPT — Yan et al. 2021 (the canonical baseline)

The blueprint that everything autoregressive builds on.

- **Stage 1** — 3D VQ-VAE with 3D convolutions + axial self-attention learns **discrete latent tokens** for video clips.
- **Stage 2** — a GPT-style transformer autoregressively models the token sequence with spatiotemporal position encoding.
- Competitive with GANs on BAIR and UCF-101 — the first strong "video = tokens + language model" result.

> **Reference**: Yan et al. "VideoGPT: Video Generation using VQ-VAE and Transformers", arXiv:2104.10157.

### 3.4 MAGVIT-v2 — Yu et al. ICLR 2024: "Language Model Beats Diffusion"

A landmark: a pure LLM over visual tokens **beats diffusion** on ImageNet and Kinetics.

Two key ideas:
- A **joint image + video tokenizer** — the same VQ vocabulary for still images and videos.
- **Lookup-Free Quantization (LFQ)** — each latent dimension is independently binarised, allowing the codebook to scale to $2^{18}$ entries, previously infeasible.

Punchline from the paper title: **"The tokenizer is key."** Given a good enough discrete video tokenizer, a plain autoregressive LLM is competitive with (or better than) a diffusion model.

> **Reference**: Yu et al. "Language Model Beats Diffusion — Tokenizer is Key for Visual Generation", arXiv:2310.05737, ICLR 2024.

### 3.5 Genie 2 — DeepMind (Dec 2024): autoregressive latent **diffusion** world model

Genie 2 is the hybrid. It is **autoregressive across frames** but **diffusion within each frame**:

<center>
<svg width="700" height="230" xmlns="http://www.w3.org/2000/svg" font-family="Arial, sans-serif" font-size="11">
  <defs><marker id="aG" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#666"/></marker></defs>
  <text x="350" y="18" text-anchor="middle" font-weight="bold" font-size="13" fill="#333">Genie 2 — AR across frames, diffusion within each frame</text>
  <!-- three frames, each a diffusion block -->
  <g transform="translate(40,50)">
    <rect x="0" y="0" width="170" height="115" rx="8" fill="#fff9f0" stroke="#f9ab00" stroke-width="1.5"/>
    <text x="85" y="18" text-anchor="middle" font-weight="bold" fill="#e37400">latent frame 1</text>
    <!-- mini denoising chain -->
    <circle cx="20" cy="55" r="8" fill="#f3f4f6" stroke="#bbb"/>
    <line x1="30" y1="55" x2="50" y2="55" stroke="#666" stroke-width="1" marker-end="url(#aG)"/>
    <circle cx="65" cy="55" r="8" fill="#e0e7ff" stroke="#7c4dff"/>
    <line x1="75" y1="55" x2="95" y2="55" stroke="#666" stroke-width="1" marker-end="url(#aG)"/>
    <circle cx="110" cy="55" r="8" fill="#cfe2ff" stroke="#4285f4"/>
    <line x1="120" y1="55" x2="140" y2="55" stroke="#666" stroke-width="1" marker-end="url(#aG)"/>
    <circle cx="155" cy="55" r="8" fill="#a8d3ff" stroke="#1a73e8"/>
    <text x="85" y="82" text-anchor="middle" fill="#555" font-size="10">diffusion denoising</text>
    <text x="85" y="100" text-anchor="middle" fill="#888" font-size="9">(not quantized tokens)</text>
  </g>
  <line x1="215" y1="107" x2="245" y2="107" stroke="#ea4335" stroke-width="1.8" marker-end="url(#aG)"/>
  <text x="230" y="98" text-anchor="middle" fill="#c5221f" font-size="10">AR</text>
  <g transform="translate(250,50)">
    <rect x="0" y="0" width="170" height="115" rx="8" fill="#fff9f0" stroke="#f9ab00" stroke-width="1.5"/>
    <text x="85" y="18" text-anchor="middle" font-weight="bold" fill="#e37400">latent frame 2</text>
    <circle cx="20" cy="55" r="8" fill="#f3f4f6" stroke="#bbb"/>
    <line x1="30" y1="55" x2="50" y2="55" stroke="#666" stroke-width="1" marker-end="url(#aG)"/>
    <circle cx="65" cy="55" r="8" fill="#e0e7ff" stroke="#7c4dff"/>
    <line x1="75" y1="55" x2="95" y2="55" stroke="#666" stroke-width="1" marker-end="url(#aG)"/>
    <circle cx="110" cy="55" r="8" fill="#cfe2ff" stroke="#4285f4"/>
    <line x1="120" y1="55" x2="140" y2="55" stroke="#666" stroke-width="1" marker-end="url(#aG)"/>
    <circle cx="155" cy="55" r="8" fill="#a8d3ff" stroke="#1a73e8"/>
    <text x="85" y="82" text-anchor="middle" fill="#555" font-size="10">conditioned on frame 1</text>
    <text x="85" y="100" text-anchor="middle" fill="#888" font-size="9">+ action a₁</text>
  </g>
  <line x1="425" y1="107" x2="455" y2="107" stroke="#ea4335" stroke-width="1.8" marker-end="url(#aG)"/>
  <text x="440" y="98" text-anchor="middle" fill="#c5221f" font-size="10">AR</text>
  <g transform="translate(460,50)">
    <rect x="0" y="0" width="170" height="115" rx="8" fill="#fff9f0" stroke="#f9ab00" stroke-width="1.5"/>
    <text x="85" y="18" text-anchor="middle" font-weight="bold" fill="#e37400">latent frame 3</text>
    <circle cx="20" cy="55" r="8" fill="#f3f4f6" stroke="#bbb"/>
    <line x1="30" y1="55" x2="50" y2="55" stroke="#666" stroke-width="1" marker-end="url(#aG)"/>
    <circle cx="65" cy="55" r="8" fill="#e0e7ff" stroke="#7c4dff"/>
    <line x1="75" y1="55" x2="95" y2="55" stroke="#666" stroke-width="1" marker-end="url(#aG)"/>
    <circle cx="110" cy="55" r="8" fill="#cfe2ff" stroke="#4285f4"/>
    <line x1="120" y1="55" x2="140" y2="55" stroke="#666" stroke-width="1" marker-end="url(#aG)"/>
    <circle cx="155" cy="55" r="8" fill="#a8d3ff" stroke="#1a73e8"/>
    <text x="85" y="82" text-anchor="middle" fill="#555" font-size="10">conditioned on 1..2</text>
    <text x="85" y="100" text-anchor="middle" fill="#888" font-size="9">+ action a₂</text>
  </g>
  <!-- Dynamics transformer bar -->
  <rect x="40" y="180" width="590" height="30" rx="6" fill="#fce4ec" stroke="#e91e63" stroke-width="1.5"/>
  <text x="335" y="199" text-anchor="middle" fill="#880e4f" font-weight="bold">Causal-masked dynamics transformer · CFG on actions</text>
</svg>
</center>

- Each latent frame is **denoised by diffusion** (not quantised) — hybrid approach.
- Latent frames are fed autoregressively to a large **causal-masked dynamics transformer** — like an LLM whose tokens are whole diffusion-generated frames.
- **Classifier-free guidance on actions** → controllable simulation.
- Generates consistent **10–60-second 3D worlds** from a single prompt.

Genie 2 is the current clearest example of the diffusion × autoregressive merge.

### 3.6 GameNGen — Valevski et al. ICLR 2025: real-time game engine

- Fine-tuned **Stable Diffusion 1.4** as a **next-frame predictor** conditioned on the user's action + the last $N$ frames.
- Simulates **DOOM at 20 fps** on a single TPU.
- Image quality indistinguishable from the real engine in user studies.
- An RL agent is used to generate 10M env-step training trajectories.
- Limitation: memory window is $\sim 3$ s — architectural bottleneck for long-horizon consistency.

> **Reference**: Valevski et al. "Diffusion Models Are Real-Time Game Engines", arXiv:2408.14837, ICLR 2025.

### 3.7 Summary: Parallel vs Autoregressive — When to Use Which

**Parallel diffusion** wins when you want:
- A short, highly polished clip (e.g. a 5-s ad).
- Strong global temporal coherence.
- Maximum sample quality per compute.

**Autoregressive** (possibly with diffusion inside each step) wins when you want:
- Streaming / unbounded length.
- **Interactivity** — condition on user actions, run a game engine, simulate an environment.
- Compute on demand (one frame at a time) instead of one big inference.

[**Transition**]: *Now let's actually run a video model.*

## 4. Practice — Generating Videos

We run two models that reliably fit on **free Colab T4 (16 GB VRAM)**:

1. **Stable Video Diffusion** — the 2023 3D-U-Net + temporal-layers recipe (image-to-video).
2. **Wan 2.1 T2V-1.3B** — our main text-to-video demo (Apache 2.0, $\sim$8 GB bf16 with CPU offload).

Both pipelines use `enable_model_cpu_offload()` — this keeps the T5 text encoder and 3D VAE on CPU when idle, so only the active sub-module sits on the 16 GB GPU. Between every load we clean GPU state; on T4 this is mandatory.

**Why not LTX-Video, CogVideoX, HunyuanVideo?**
- HunyuanVideo (13B) and Mochi (10B) need ≥ 24 GB VRAM even with offload.
- CogVideoX-2B + T5-XXL is right at the T4 limit and frequently OOMs on 81-frame outputs.
- LTX-Video (`Lightricks/LTX-Video`) only fits on T4 with Q8 quantisation (`optimum-quanto`); without quant it exhausts VRAM during the T5-XXL + DiT forward. See the [LTX-Video docs](https://huggingface.co/Lightricks/LTX-Video) for the quantized recipe.

For any of these you want a single **A100 / H100** or a paid Colab Pro tier with A100.

### 4.1 Demo 1 — Stable Video Diffusion (historical 3D-U-Net, Image-to-Video)

This is the 2023 recipe: a frozen image LDM (Stable Diffusion 2.1) plus inserted temporal conv + attention layers, fine-tuned on a curated video dataset. It represents **everything that came before the DiT-for-video shift**.

In [ ]:
from diffusers import StableVideoDiffusionPipeline

pipe_svd = StableVideoDiffusionPipeline.from_pretrained(
    "stabilityai/stable-video-diffusion-img2vid-xt",
    torch_dtype=torch.float16,
    variant="fp16",
).to("cuda")

# Memory-friendly: offload text encoder / VAE when idle
pipe_svd.enable_model_cpu_offload()

print(f"SVD U-Net params: {sum(p.numel() for p in pipe_svd.unet.parameters())/1e6:.0f}M")

In [ ]:
# Use the rocket reference image from HF docs
image = load_image(
    "https://huggingface.co/datasets/huggingface/documentation-images/"
    "resolve/main/diffusers/svd/rocket.png"
)
image = image.resize((1024, 576))

generator = torch.Generator("cuda").manual_seed(SEED)
frames = pipe_svd(
    image,
    decode_chunk_size=8,       # decode 8 frames at a time to save VRAM
    generator=generator,
    num_frames=14,
    motion_bucket_id=127,      # higher = more motion
    noise_aug_strength=0.02,
).frames[0]

export_to_video(frames, "svd_rocket.mp4", fps=7)
media.show_video(media.read_video("svd_rocket.mp4"), fps=7)

In [ ]:
# Cleanup — always between model loads
del pipe_svd
torch.cuda.empty_cache()
gc.collect()
print(f"After SVD cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")

**What to notice in the SVD output**:
- Clear *camera* motion (this is what the temporal layers primarily learned).
- Very limited *scene* dynamics — the rocket does not light, exhaust does not evolve, because the temporal layers were bolted onto a frozen image model and the model has no strong world-dynamics prior.
- This is the **characteristic artefact of the 3D-U-Net generation**. Modern DiT-based models (Wan, CogVideoX, Hunyuan) look qualitatively different — richer internal motion, fewer "slideshow" moments.

### 4.2 Demo 2 — Wan 2.1 T2V-1.3B (main live demo)

`Wan2.1-T2V-1.3B` is the centrepiece of this seminar: it is the **only model in our list that truly fits free Colab T4** (~8 GB bf16). Its design — DiT + cross-attention + 3D VAE + T5 + Flow Matching — is the mainstream open-source consensus of 2025.

In [ ]:
from transformers import UMT5EncoderModel, BitsAndBytesConfig as TBnbConfig
from diffusers import WanPipeline, AutoencoderKLWan

model_id = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"

# --- TEXT ENCODER ----------------------------------------------------------
# UMT5-XXL is ~11 GB in bf16 — bigger than Colab free RAM (~13 GB).
# We need TWO things, not one, to avoid the OOM-kill you saw:
#   1. `load_in_4bit=True`  → final model is ~3 GB on GPU.
#   2. `device_map="auto"`  → accelerate streams the checkpoint directly into
#                             bnb 4-bit on the GPU. Without this flag, the
#                             fp16 copy is fully materialised in RAM first,
#                             which is what killed the kernel.
text_encoder = UMT5EncoderModel.from_pretrained(
    model_id,
    subfolder="text_encoder",
    quantization_config=TBnbConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
    ),
    torch_dtype=torch.float16,
    device_map="auto",
)

# --- VAE -------------------------------------------------------------------
# fp32 per Wan docs; low_cpu_mem_usage loads chunk-by-chunk so no spike.
vae = AutoencoderKLWan.from_pretrained(
    model_id, subfolder="vae",
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True,
)

# --- DiT -------------------------------------------------------------------
# fp16, not bf16 — T4 has no native bf16.
pipe_wan = WanPipeline.from_pretrained(
    model_id,
    text_encoder=text_encoder,
    vae=vae,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

# The 4-bit text encoder already lives on GPU (~3 GB). Offload DiT + VAE
# to CPU between forwards; enable VAE slicing + tiling for the decode peak.
pipe_wan.enable_model_cpu_offload()
pipe_wan.vae.enable_slicing()
pipe_wan.vae.enable_tiling()

print(f"Wan 2.1 transformer params: "
      f"{sum(p.numel() for p in pipe_wan.transformer.parameters())/1e9:.2f}B")

In [ ]:
prompt = (
    "A cat wearing a wizard hat casts a glowing rainbow spell in a dusty "
    "medieval library, cinematic lighting, highly detailed"
)
negative = "blurry, low quality, distorted, static, text, watermark"

generator = torch.Generator("cuda").manual_seed(SEED)
result = pipe_wan(
    prompt=prompt,
    negative_prompt=negative,
    height=480,
    width=832,
    num_frames=81,                # ~5 s at 16 fps
    num_inference_steps=30,
    guidance_scale=5.0,           # lower than SD 1.5 — typical for video FM
    generator=generator,
).frames[0]

export_to_video(result, "wan_cat.mp4", fps=16)
media.show_video(media.read_video("wan_cat.mp4"), fps=16)

**Observations**:
- Character identity is **stable** across all 81 frames — no face morphing. This is the 3D causal VAE with anti-flicker training doing its job.
- Rich internal dynamics: the hat moves with the head, the spell evolves, the camera drifts. Compare with SVD above.
- The prompt is followed well despite T5-only text conditioning — Wan's training data and captioning pipeline are strong.

### 4.3 Demo 3 — Parameter Exploration

With the same pipeline loaded, sweep the main knobs and observe the trade-offs.

In [ ]:
# Fixed prompt / seed, vary inference steps
prompt_fixed = "A drone shot flying over a foggy mountain ridge at sunrise, cinematic"
negative_fixed = "blurry, low quality, static"

sweeps = [("steps=15", dict(num_inference_steps=15, guidance_scale=5.0)),
          ("steps=30", dict(num_inference_steps=30, guidance_scale=5.0)),
          ("cfg=3.0",  dict(num_inference_steps=30, guidance_scale=3.0)),
          ("cfg=7.5",  dict(num_inference_steps=30, guidance_scale=7.5))]

videos = {}
for name, kwargs in sweeps:
    generator = torch.Generator("cuda").manual_seed(SEED)
    out = pipe_wan(
        prompt=prompt_fixed,
        negative_prompt=negative_fixed,
        height=480, width=832,
        num_frames=33,                  # shorter for the sweep — ~2 s
        generator=generator,
        **kwargs,
    ).frames[0]
    path = f"wan_sweep_{name}.mp4"
    export_to_video(out, path, fps=16)
    videos[name] = path
    print(f"saved {name} → {path}")

In [ ]:
# Display the sweep side by side
for name, path in videos.items():
    print(name)
    media.show_video(media.read_video(path), fps=16)

**What to look for**:
- **Steps 15 → 30**: structure is roughly formed by step 15 (Flow Matching does a lot in few steps), but fine textures and motion coherence improve with more steps.
- **CFG 3.0 vs 7.5**: higher guidance → tighter prompt adherence but more saturation / artefacts. For video the sweet spot is **typically 4–6**, noticeably lower than the 7–9 range common for SD 1.5 images.

### 4.4 Practical Patterns — What to Remember

- **Seed reuse** (`torch.Generator("cuda").manual_seed(SEED)`) is the only way to ablate a single parameter fairly — always reset the generator before each call.
- **Guidance scale sweet spot is lower for video**: typically **4–6**, not 7–9 as in SD 1.5. Higher CFG over-saturates colours and introduces motion artefacts.
- **Number of frames is quantised**. Most open models are trained around 49 / 81 / 129 frames; going substantially longer degrades quality (attention is $O(T^2)$ — the network was never trained on that shape).
- **`fps` is a display / export choice**, not a model parameter — the model produces frames at a fixed temporal rate set by the VAE's temporal stride. `fps=16` vs `fps=24` is a post-hoc decision about playback speed.
- **VAE in float32** for stability (Wan docs); **transformer in bfloat16**. Mixing precisions like this is standard for video pipelines on consumer GPUs.

### 4.5 Memory Management Recap

In [ ]:
def free_gpu(label=""):
    """Call after every `del pipe_X` to confirm memory was actually released."""
    torch.cuda.empty_cache()
    gc.collect()
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved()  / 1e9
    print(f"[{label}] allocated={allocated:.2f} GB, reserved={reserved:.2f} GB")

free_gpu("end of §4")

## 5. Open Problems — What's Still Broken in 2026

Generation quality has shot up, but four concrete failure modes keep video diffusion from being "solved".

### 5.1 Temporal Consistency

Even in state-of-the-art models you see:
- **Object identity drift** — a character's face subtly morphs over a 5-s clip.
- **Flickering fine textures** — fur, foliage, small patterns change frame-to-frame.
- **Physics violations** — objects pass through walls, gravity flips, water flows uphill.

<center>
<svg width="700" height="200" xmlns="http://www.w3.org/2000/svg" font-family="Arial, sans-serif" font-size="11">
  <text x="350" y="18" text-anchor="middle" font-weight="bold" font-size="13" fill="#333">Characteristic failure modes of modern video diffusion</text>
  <!-- Identity drift -->
  <g transform="translate(30,45)">
    <rect x="0"   y="0" width="200" height="130" rx="8" fill="#fce8e6" stroke="#ea4335" stroke-width="1.5"/>
    <text x="100" y="20" text-anchor="middle" fill="#c5221f" font-weight="bold">① Identity drift</text>
    <!-- 4 faces -->
    <circle cx="28" cy="70" r="16" fill="#fff" stroke="#c5221f" stroke-width="1"/>
    <circle cx="24" cy="67" r="1.5" fill="#c5221f"/>
    <circle cx="32" cy="67" r="1.5" fill="#c5221f"/>
    <circle cx="72" cy="70" r="16" fill="#fff" stroke="#c5221f" stroke-width="1"/>
    <circle cx="68" cy="67" r="1.5" fill="#c5221f"/>
    <circle cx="77" cy="67" r="1.5" fill="#c5221f"/>
    <circle cx="118" cy="70" r="16" fill="#fff" stroke="#c5221f" stroke-width="1"/>
    <circle cx="113" cy="66" r="1.5" fill="#c5221f"/>
    <circle cx="122" cy="68" r="1.5" fill="#c5221f"/>
    <circle cx="165" cy="70" r="18" fill="#fff" stroke="#c5221f" stroke-width="1"/>
    <circle cx="158" cy="65" r="2" fill="#c5221f"/>
    <circle cx="170" cy="68" r="1.5" fill="#c5221f"/>
    <text x="100" y="105" text-anchor="middle" fill="#555" font-size="10">face morphs across 5 s</text>
    <text x="100" y="120" text-anchor="middle" fill="#888" font-size="9">same character, drift</text>
  </g>
  <!-- Flicker -->
  <g transform="translate(250,45)">
    <rect x="0"   y="0" width="200" height="130" rx="8" fill="#fef7e0" stroke="#f9ab00" stroke-width="1.5"/>
    <text x="100" y="20" text-anchor="middle" fill="#e37400" font-weight="bold">② Fine-texture flicker</text>
    <!-- noise squares -->
    <rect x="25" y="45" width="150" height="55" rx="4" fill="#ffecb3" stroke="#f9ab00"/>
    <g fill="#c5221f" opacity="0.6">
      <circle cx="40" cy="58" r="1.2"/><circle cx="55" cy="66" r="1.2"/><circle cx="72" cy="52" r="1.2"/>
      <circle cx="90" cy="72" r="1.2"/><circle cx="108" cy="58" r="1.2"/><circle cx="125" cy="68" r="1.2"/>
      <circle cx="145" cy="53" r="1.2"/><circle cx="162" cy="73" r="1.2"/>
    </g>
    <g fill="#34a853" opacity="0.7">
      <circle cx="45" cy="72" r="1.2"/><circle cx="68" cy="63" r="1.2"/><circle cx="85" cy="55" r="1.2"/>
      <circle cx="100" cy="82" r="1.2"/><circle cx="120" cy="50" r="1.2"/><circle cx="150" cy="77" r="1.2"/>
    </g>
    <text x="100" y="120" text-anchor="middle" fill="#555" font-size="10">fur/foliage shimmer per frame</text>
  </g>
  <!-- Physics -->
  <g transform="translate(470,45)">
    <rect x="0"   y="0" width="200" height="130" rx="8" fill="#e8f5e9" stroke="#34a853" stroke-width="1.5"/>
    <text x="100" y="20" text-anchor="middle" fill="#1b5e20" font-weight="bold">③ Physics violations</text>
    <line x1="30" y1="95" x2="170" y2="95" stroke="#1b5e20" stroke-width="1.5"/>
    <rect x="90" y="40" width="20" height="55" fill="#8d6e63" stroke="#4e342e"/>
    <!-- Ball going through wall -->
    <circle cx="55"  cy="70" r="8" fill="#c5221f"/>
    <circle cx="78"  cy="70" r="8" fill="#c5221f" opacity="0.7"/>
    <circle cx="100" cy="70" r="8" fill="#c5221f" opacity="0.45"/>
    <circle cx="125" cy="70" r="8" fill="#c5221f" opacity="0.7"/>
    <circle cx="148" cy="70" r="8" fill="#c5221f"/>
    <text x="100" y="120" text-anchor="middle" fill="#555" font-size="10">objects through walls, reversed gravity</text>
  </g>
</svg>
</center>

**Root cause**: per-frame noise is sampled independently during training. Temporal correlation has to be *learned* — it is not structurally enforced.

**Current mitigations**:
- 3D causal VAEs trained with temporal-smoothness reconstruction losses (CogVideoX).
- Joint space-time attention (no factorization) — more expensive but structurally tighter (Kling, Mochi).
- Bigger models with more diverse training data.

### 5.2 The Long-Video Wall

Full 3D attention cost is **quadratic in the number of spatio-temporal tokens**:
$$\text{cost} \;=\; O\bigl((T \cdot H \cdot W)^2\bigr)$$

For 5 s at 720p with a HunyuanVideo-class model, **attention alone is $\sim$800 of 950 seconds** of inference time. Most open models cap at $\sim 5{-}10$ s clips. Going longer requires sparsity.

<center>
<svg width="620" height="260" xmlns="http://www.w3.org/2000/svg" font-family="Arial, sans-serif" font-size="11">
  <text x="310" y="18" text-anchor="middle" font-weight="bold" font-size="13" fill="#333">Attention cost grows quadratically with clip length</text>
  <!-- Axes -->
  <line x1="60"  y1="220" x2="580" y2="220" stroke="#666" stroke-width="1.2"/>
  <line x1="60"  y1="40"  x2="60"  y2="220" stroke="#666" stroke-width="1.2"/>
  <text x="320" y="245" text-anchor="middle" fill="#333" font-size="11">video length (seconds)</text>
  <text x="40"  y="130" text-anchor="middle" fill="#333" font-size="11" transform="rotate(-90 40,130)">attention cost (relative)</text>
  <!-- x ticks -->
  <text x="80"  y="236" text-anchor="middle" fill="#888" font-size="10">1</text>
  <text x="180" y="236" text-anchor="middle" fill="#888" font-size="10">3</text>
  <text x="280" y="236" text-anchor="middle" fill="#888" font-size="10">5</text>
  <text x="380" y="236" text-anchor="middle" fill="#888" font-size="10">10</text>
  <text x="480" y="236" text-anchor="middle" fill="#888" font-size="10">30</text>
  <text x="560" y="236" text-anchor="middle" fill="#888" font-size="10">60</text>
  <!-- Parabola approximation: y ~ x^2 -->
  <path d="M 80 215 Q 180 210 280 195 Q 380 160 480 95 Q 540 60 580 45"
        fill="none" stroke="#ea4335" stroke-width="2.2"/>
  <text x="560" y="60" text-anchor="end" fill="#c5221f" font-size="10">O((T·H·W)²)</text>
  <!-- Typical open-source window -->
  <rect x="80" y="175" width="200" height="45" fill="#e8f5e9" opacity="0.55" stroke="#34a853" stroke-width="1.3" stroke-dasharray="4,3"/>
  <text x="180" y="195" text-anchor="middle" fill="#1b5e20" font-size="10" font-weight="bold">open-source safe zone</text>
  <text x="180" y="210" text-anchor="middle" fill="#1b5e20" font-size="10">≤ 10 s (5–10 s typical)</text>
  <!-- Sparse methods -->
  <path d="M 80 215 Q 200 203 320 186 Q 440 170 560 150" fill="none" stroke="#34a853" stroke-width="2" stroke-dasharray="5,3"/>
  <text x="560" y="165" text-anchor="end" fill="#1b5e20" font-size="10">sparse attn (STA, MoC, FreeSwim) — near-linear</text>
  <!-- Kling marker -->
  <circle cx="500" cy="75" r="5" fill="#c5221f"/>
  <text x="510" y="80" fill="#c5221f" font-size="10">Kling ≤ 2 min (closed)</text>
</svg>
</center>

**Active research directions**:

| Method | Idea | Reference |
|---|---|---|
| **Sliding Tile Attention (STA)** | Hardware-aware tile-local attention in time/space | arXiv:2502.04507 |
| **FreeSwim** | Inward sliding window preserving the training receptive field | 2025 |
| **Mixture of Contexts (MoC)** | Sparse retrieval of relevant chunks + anchors per query | arXiv:2508.21058 |
| **Block-autoregressive rollouts** | Generate in blocks of frames, condition on previous blocks; self-forcing distillation | 2024–25 |
| **Lumiere-style** | Run temporal attention only at the coarsest space-time scale of a pyramid | 2024 |

[**Accent**] This is the **single biggest bottleneck** in the field right now. Every open question — long narratives, interactive simulation, high-fps generation — eventually bottoms out on this one.

### 5.3 Motion Control

Describing motion in text is fundamentally lossy. Current control tracks:

- **Text-based** ("the camera pans left") — unreliable, imprecise.
- **Trajectory-based** — draw curves for object motion (DragNUWA, MotionCtrl).
- **Camera control** — explicit camera pose inputs (CameraCtrl, MotionDirector).
- **Keyframe conditioning** — specify start / middle / end frames.
- **Physics priors** — Genie 2 learns **latent actions** from gameplay data; these give surprisingly crisp control.

### 5.4 Beyond Generation — Controllable World Models

Pure-quality generation is now close to saturation on standard benchmarks. The next frontier is **controllable simulation**:

- **Interactive world models** — video conditioned on a user action stream (Genie 2, GameNGen).
- **Applications**: RL environment simulation, game engines, robotics sim-to-real, virtual production.
- **Open question**: can we train a single foundation model that is both a good generator *and* a good simulator?

### 5.5 Current Frontier (early 2026)

- **Joint audio-video generation** (Veo 3, Wan-Audio): unified token sequence for pixels and waveforms.
- **Real-time generation**: LTX-Video and Wan 2.1-1.3B are approaching faster-than-real-time on a single 4090.
- **Consumer-grade models**: HunyuanVideo-1.5 (8.3B) explicitly targets $\leq 16$ GB GPUs.
- **MoE along the denoising trajectory** (Wan 2.2): specialised experts for different noise levels.

### 5.6 Connection to the Rest of the Course

| Seminar / Lecture | Image / theory side | Video analogue |
|---|---|---|
| Seminar 9 | SD 1.5 U-Net | VDM + factorized attention |
| Seminar 10 | ControlNet / IP-Adapter / LoRA | Camera control, DragNUWA, MotionDirector |
| Seminar 11 | DiT / MMDiT / FLUX | Sora, CogVideoX, HunyuanVideo, Wan |
| Lectures 11–12 | Flow Matching, Conditional FM | Every modern video model uses v-prediction / RF |
| Lectures 13–14 (discrete diffusion) | Masked diffusion language models | **MAGVIT-v2 — LLM over discrete video tokens** |

The next block of lectures will bring diffusion and language models together for us — and as MAGVIT-v2 shows, that merge is already happening on the video side.

## 6. Summary — Image Diffusion (Seminar 11) Side by Side with Video Diffusion

| Axis | Image (Seminar 11) | Video (this seminar) |
|---|---|---|
| **Backbone** | UNet → DiT → MMDiT → FLUX | 3D U-Net → DiT → MMDiT → Dual+Single stream → MoE-along-noise |
| **VAE** | 2D VAE | 3D causal VAE ($8\times8\times4$); extreme LTX 1:192 |
| **Text encoder** | CLIP → Dual CLIP → Triple (+T5) | T5-XXL **or** MLLM (llava-llama-3-8b) |
| **Objective** | DDPM $\epsilon$ → Rectified Flow v | Rectified Flow v (universal in 2024+) |
| **Positional enc.** | Learned / Fourier → RoPE | **3D RoPE** on $(t, h, w)$ |
| **Attention** | Cross-attn → Joint self-attn | Factorized space-time → Joint 3D |
| **Scale** | 860M → 12B | 1.3B (Wan-small) → 27B (Wan 2.2 MoE) |
| **Open problems** | Text rendering, consistency | **Long videos, motion control, temporal consistency, physics** |

<center>
<svg width="720" height="340" xmlns="http://www.w3.org/2000/svg" font-family="Arial, sans-serif" font-size="11">
  <text x="360" y="20" text-anchor="middle" font-weight="bold" font-size="14" fill="#333">Seminar 11 (images) ⟷ Seminar 12 (video) — same story, +1 dimension</text>
  <!-- Header labels -->
  <rect x="30"  y="40" width="330" height="30" rx="6" fill="#e8f0fe" stroke="#4285f4" stroke-width="1.5"/>
  <text x="195" y="60" text-anchor="middle" fill="#1a73e8" font-weight="bold">Image diffusion — 2022–2023</text>
  <rect x="370" y="40" width="320" height="30" rx="6" fill="#fce8e6" stroke="#ea4335" stroke-width="1.5"/>
  <text x="530" y="60" text-anchor="middle" fill="#c5221f" font-weight="bold">Video diffusion — 2024–2025</text>
  <!-- Row 1: architecture -->
  <g transform="translate(30,85)">
    <rect x="0"   y="0" width="330" height="32" rx="4" fill="#fef7e0" stroke="#f9ab00"/>
    <text x="165" y="20" text-anchor="middle" fill="#e37400" font-size="11">UNet → DiT → MMDiT → FLUX</text>
    <rect x="340" y="0" width="320" height="32" rx="4" fill="#fef7e0" stroke="#f9ab00"/>
    <text x="500" y="20" text-anchor="middle" fill="#e37400" font-size="11">3D UNet → DiT → MMDiT → dual+single → MoE-noise</text>
  </g>
  <!-- Row 2: VAE -->
  <g transform="translate(30,125)">
    <rect x="0"   y="0" width="330" height="32" rx="4" fill="#e8f5e9" stroke="#34a853"/>
    <text x="165" y="20" text-anchor="middle" fill="#1b5e20" font-size="11">2D VAE (SD, 4–16 channels)</text>
    <rect x="340" y="0" width="320" height="32" rx="4" fill="#e8f5e9" stroke="#34a853"/>
    <text x="500" y="20" text-anchor="middle" fill="#1b5e20" font-size="11">3D causal VAE 8×8×4 · LTX 1:192</text>
  </g>
  <!-- Row 3: Text -->
  <g transform="translate(30,165)">
    <rect x="0"   y="0" width="330" height="32" rx="4" fill="#ede7f6" stroke="#7c4dff"/>
    <text x="165" y="20" text-anchor="middle" fill="#4a148c" font-size="11">CLIP → CLIP+OpenCLIP → +T5</text>
    <rect x="340" y="0" width="320" height="32" rx="4" fill="#ede7f6" stroke="#7c4dff"/>
    <text x="500" y="20" text-anchor="middle" fill="#4a148c" font-size="11">T5-XXL · MLLM llava-llama-3-8b</text>
  </g>
  <!-- Row 4: Objective -->
  <g transform="translate(30,205)">
    <rect x="0"   y="0" width="330" height="32" rx="4" fill="#fce4ec" stroke="#e91e63"/>
    <text x="165" y="20" text-anchor="middle" fill="#880e4f" font-size="11">DDPM ε → Rectified Flow v</text>
    <rect x="340" y="0" width="320" height="32" rx="4" fill="#fce4ec" stroke="#e91e63"/>
    <text x="500" y="20" text-anchor="middle" fill="#880e4f" font-size="11">Flow Matching v (universal 2024+)</text>
  </g>
  <!-- Row 5: PE -->
  <g transform="translate(30,245)">
    <rect x="0"   y="0" width="330" height="32" rx="4" fill="#e8f0fe" stroke="#4285f4"/>
    <text x="165" y="20" text-anchor="middle" fill="#1a73e8" font-size="11">learned / Fourier → RoPE</text>
    <rect x="340" y="0" width="320" height="32" rx="4" fill="#e8f0fe" stroke="#4285f4"/>
    <text x="500" y="20" text-anchor="middle" fill="#1a73e8" font-size="11">3D RoPE over (t, h, w)</text>
  </g>
  <!-- Lag banner -->
  <rect x="100" y="295" width="520" height="34" rx="8" fill="#fff9f0" stroke="#f9ab00" stroke-width="1.5"/>
  <text x="360" y="316" text-anchor="middle" fill="#e37400" font-weight="bold" font-size="12">video recapitulates image with ~6-month lag on every axis</text>
</svg>
</center>

[**Final takeaway**]

Video diffusion in 2024–2025 **recapitulates image diffusion's 2022–2023 evolution** with a ~6-month lag. The same four axes — architecture, VAE, text encoder, training objective — play out in the same order, just applied to 4D tensors instead of 3D ones.

The next seminar block covers **discrete / masked diffusion** — the frontier where autoregressive and diffusion paradigms finally merge. MAGVIT-v2's "tokenizer + LLM beats diffusion" result is the first strong signal in that direction.

---

**References** (primary papers):

1. Ho et al. "Video Diffusion Models" (2022) — arXiv:2204.03458
2. Ho et al. "Imagen Video" (2022)
3. Singer et al. "Make-A-Video" (2022)
4. Blattmann et al. "Align Your Latents / Video LDM" (2023)
5. Blattmann et al. "Stable Video Diffusion" (2023)
6. Bar-Tal et al. "Lumiere" (2024)
7. Brooks et al. "Sora technical report" — OpenAI (Feb 2024)
8. Yang et al. "CogVideoX" — arXiv:2408.06072 (ICLR 2025)
9. Kong et al. "HunyuanVideo: A Systematic Framework for Large Video Generation Model" — Tencent (Dec 2024)
10. Genmo "Mochi 1" (Oct 2024)
11. HaCohen et al. "LTX-Video: Realtime Video Latent Diffusion" — arXiv:2501.00103 (2025)
12. Wan Team "Wan 2.1" (Feb 2025), "Wan 2.2" (July 2025)
13. DeepMind "Veo 3 technical report" (2025)
14. Yu et al. "Language Model Beats Diffusion — Tokenizer is Key" (MAGVIT-v2) — arXiv:2310.05737
15. Yan et al. "VideoGPT" — arXiv:2104.10157
16. Bruce et al. "Genie" — arXiv:2402.15391
17. Valevski et al. "GameNGen: Diffusion Models Are Real-Time Game Engines" — arXiv:2408.14837
18. VideoRoPE — arXiv:2502.05173
19. Sliding Tile Attention — arXiv:2502.04507
20. Mixture of Contexts — arXiv:2508.21058